# CNN-EXP-046-B — recovery / restart-safe Stage 2

This notebook continues the existing **run_001** after the kernel restart during Stage 2.

Scientific state at recovery point:

- Stage 1 train-only population/sampling is accepted and **not recomputed**;
- no head training, train-contig CV, config selection, or validation evaluation has started;
- the existing feature cache is treated as incomplete and resumed through `filled_u8.dat`;
- the first four legacy filled rows are deliberately invalidated once and recomputed, because the original writer did not flush data before setting the journal bit;
- WIDTH512 inference batch is reduced from 2 to 1;
- regional mean/max pooling is moved from DirectML to CPU after copying only the required 36×32 feature slice;
- cache writes are checkpointed data-first / journal-last, so a crash cannot mark an unflushed row complete.

A separate preregistration amendment is written before CV. The scientific model design, R32 grid, sample manifest, loss, architecture, alpha grid, CV split, and validation protocol are unchanged. The training schedule is amended **before any head training/CV result exists** to: 5% optimizer-step warm-up → peak LR 1e-3 → cosine decay to 1e-4 over the same four epochs.


In [1]:
from pathlib import Path
import gc
import hashlib
import inspect
import json
import math
import os
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_directml
from IPython.display import display

from sklearn.metrics import roc_auc_score

# ============================================================
# HARD ENVIRONMENT / PROJECT GUARDS
# ============================================================
EXPECTED_PYTHON = os.path.normcase(os.path.abspath(
    r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe"
))
ACTUAL_PYTHON = os.path.normcase(os.path.abspath(sys.executable))
assert ACTUAL_PYTHON == EXPECTED_PYTHON, (
    f"Wrong interpreter: {sys.executable}\nExpected: {EXPECTED_PYTHON}"
)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src" / "ml_project").is_dir()
)
EXPECTED_PROJECT_ROOT = Path(r"D:\Projects\EPIC\EPIC").resolve()
assert os.path.normcase(str(PROJECT_ROOT)) == os.path.normcase(str(EXPECTED_PROJECT_ROOT)), (
    f"Wrong project root: {PROJECT_ROOT}"
)

AGENTS_PATH = PROJECT_ROOT.parent / "AGENTS.md"
assert AGENTS_PATH.is_file(), f"Missing required {AGENTS_PATH}"
AGENTS_SHA256 = hashlib.sha256(AGENTS_PATH.read_bytes()).hexdigest()
EXPECTED_AGENTS_SHA256 = "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3"
assert AGENTS_SHA256 == EXPECTED_AGENTS_SHA256, (
    "AGENTS.md changed since the accepted EXP046 oracle/frontier run. "
    "Review the new instructions before continuing.\n"
    f"Current: {AGENTS_SHA256}\nExpected: {EXPECTED_AGENTS_SHA256}"
)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

# ============================================================
# PRE-REGISTERED SCIENTIFIC CONSTANTS + RECOVERY AMENDMENT
# ============================================================
EXPERIMENT_ID = "CNN-EXP-046-B"
REGION_SIZE = 32
GRID_ORIGIN = 0
REGIONS_PER_TILE = 32
NEIGHBOR_REGIONS = 2
CONTEXT_REGIONS = 5
SCORE_STAT_CHANNELS = 4
BACKBONE_POOL_CHANNELS = 1024
VARIANT_CHANNELS = {"A": SCORE_STAT_CHANNELS, "B": SCORE_STAT_CHANNELS + BACKBONE_POOL_CHANNELS}

ALPHAS = (0.0, 0.25, 0.5, 1.0, 2.0)
NEG_PER_STRATUM_PER_CONTIG = 2_500
NEGATIVE_STRATA = ("0_1", "1_5", "5_15", "15_100")
SAMPLE_SEED = 460_600
HEAD_SEED = 460_601
HEAD_EPOCHS = 4
HEAD_BATCH_SIZE = 256
HEAD_PEAK_LR = 1e-3
HEAD_FINAL_LR = 1e-4
HEAD_WARMUP_FRACTION = 0.05
HEAD_WEIGHT_DECAY = 1e-4
CACHE_INFERENCE_BATCH_SIZE = 1
EVAL_INFERENCE_BATCH_SIZE = 1
CACHE_FLUSH_ROWS = 128
STAT_CHUNK = 128
MAX_SAMPLE_CONTEXT_CACHE_GIB = 4.0
MIN_FREE_DISK_MARGIN_GIB = 8.0

TRAINING_SCHEDULE = {
    "type": "optimizer_step_warmup_plus_cosine",
    "epochs": HEAD_EPOCHS,
    "warmup_fraction": HEAD_WARMUP_FRACTION,
    "peak_lr": HEAD_PEAK_LR,
    "final_lr": HEAD_FINAL_LR,
}

BUDGET_MULTIPLIERS = (1, 2, 5, 10)
REGIONAL_RECALL_DIAGNOSTIC_TARGETS = (0.99, 0.98, 0.95)

EXPECTED_TRAIN_VALID_REGIONS = 7_284_784
EXPECTED_TRAIN_ACTIVE_REGIONS = 83_887
EXPECTED_TRAIN_EMPTY_REGIONS = 7_200_897
EXPECTED_TRAIN_SAMPLE_ROWS = 173_887
EXPECTED_TRAIN_SAMPLE_ACTIVE = 83_887
EXPECTED_TRAIN_SAMPLE_EMPTY = 90_000
EXPECTED_TRAIN_LOWTP = 42_613

EXPECTED_VALIDATION_POSITION_STRAND = 83_807_486
EXPECTED_VALIDATION_POSITIVES = 56_205
EXPECTED_BASELINE_AP5 = 0.09763009437871803
EXPECTED_ORACLE32_AP5 = 0.23831825797792802
EXPECTED_VALIDATION_CONTIG_AP5 = {
    "NC_064034.1": 0.09569052125003817,
    "NC_064041.1": 0.09140733554044979,
    "NC_064042.1": 0.10520206334231234,
}
AP_TOL = 5e-6
SUBMISSION_LEVELS = 100_001

EXP045_RUN_DIR = (
    PROJECT_ROOT / "artifacts" / "experiments" /
    "CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029" / "run_001"
)
CHECKPOINT_PATH = EXP045_RUN_DIR / "width512_50k_final.pt"
assert CHECKPOINT_PATH.is_file(), f"Missing checkpoint: {CHECKPOINT_PATH}"

EXP046_RUN_DIR = (
    PROJECT_ROOT / "artifacts" / "experiments" /
    "CNN_EXP046_REGIONAL_ACTIVITY_GATE" / "run_001"
)
FRONTIER_CSV = EXP046_RUN_DIR / "feasibility_frontier_r32" / "r32_target_ap_crossings.csv"
ORACLE_JSON = EXP046_RUN_DIR / "oracle_regional_gate_summary.json"
FIXED_VALIDATION_LOW_TP = EXP046_RUN_DIR / "fixed_low_tp_coordinates.csv"
FIXED_VALIDATION_HARD_FP = EXP046_RUN_DIR / "fixed_hard_fp_coordinates.csv"
for p in (FRONTIER_CSV, ORACLE_JSON, FIXED_VALIDATION_LOW_TP, FIXED_VALIDATION_HARD_FP):
    assert p.is_file(), f"Missing accepted EXP046 artifact: {p}"

frontier_crossings = pd.read_csv(FRONTIER_CSV)
frontier_099 = frontier_crossings[np.isclose(frontier_crossings["active_region_recall_target"], 0.99)].iloc[0]
assert abs(float(frontier_099["median_first_empty_suppression_fraction"]) - 0.10) < 1e-5
assert abs(float(frontier_099["median_ap1"]) - 0.20308937905110025) < 1e-12
accepted_oracle = json.loads(ORACLE_JSON.read_text(encoding="utf-8"))
assert int(accepted_oracle["primary_region_size"]) == REGION_SIZE
assert abs(float(accepted_oracle["primary_oracle_ap1"]) - EXPECTED_ORACLE32_AP5) <= 1e-12

RUN_DIR = (
    PROJECT_ROOT / "artifacts" / "experiments" /
    "CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32" / "run_001"
)
assert RUN_DIR.is_dir(), f"Recovery requires existing run_001: {RUN_DIR}"
CACHE_DIR = RUN_DIR / "_train_sample_cache"
FOLD_MODEL_DIR = RUN_DIR / "fold_models"
CV_RAW_DIR = RUN_DIR / "_cv_raw_cache"
VALIDATION_RAW_DIR = RUN_DIR / "_validation_raw_cache"
assert CACHE_DIR.is_dir(), f"Missing interrupted Stage-2 cache: {CACHE_DIR}"
if FOLD_MODEL_DIR.exists():
    assert not any(FOLD_MODEL_DIR.iterdir()), "fold_models is not empty: CV/head training may already have started"
else:
    FOLD_MODEL_DIR.mkdir(parents=False, exist_ok=False)
assert not CV_RAW_DIR.exists(), "CV raw cache exists; stop and inspect before recovery"
assert not VALIDATION_RAW_DIR.exists(), "Validation raw cache exists; stop and inspect before recovery"

OUTPUTS = {
    "prereg": RUN_DIR / "EXP046B_preregistration.json",
    "sample_manifest": RUN_DIR / "train_region_sample_manifest.csv",
    "sampling_summary": RUN_DIR / "train_sampling_summary.csv",
    "train_lowtp": RUN_DIR / "train_fixed_lowtp_coordinates.csv",
    "channel_stats": RUN_DIR / "sample_channel_sufficient_stats.npz",
    "cv_fold_ap": RUN_DIR / "cv_fold_ap5.csv",
    "cv_pooled_ap": RUN_DIR / "cv_pooled_ap5.csv",
    "cv_region_diag": RUN_DIR / "cv_regional_diagnostics.csv",
    "cv_budget": RUN_DIR / "cv_fixed_budget_recall.csv",
    "cv_raw_audit": RUN_DIR / "cv_selected_raw_ap_by_contig.csv",
    "selected_config": RUN_DIR / "selected_config_from_train_cv.json",
    "cv_plot": RUN_DIR / "cv_alpha_curves.png",
    "final_head": RUN_DIR / "final_regional_head.pt",
    "validation_metrics": RUN_DIR / "validation_nucleotide_metrics.csv",
    "validation_budget": RUN_DIR / "validation_fixed_budget_recall.csv",
    "validation_region_diag": RUN_DIR / "validation_regional_diagnostics.csv",
    "validation_cohorts": RUN_DIR / "validation_fixed_cohort_score_summary.csv",
    "validation_plot": RUN_DIR / "validation_contig_ap.png",
    "result_json": RUN_DIR / "EXP046B_RESULT.json",
}

# Stage 1 + original prereg must exist. No scientific result may exist yet.
for p in (OUTPUTS["prereg"], OUTPUTS["sample_manifest"], OUTPUTS["sampling_summary"], OUTPUTS["train_lowtp"], RUN_DIR / "train_region_population.json"):
    assert p.is_file(), f"Missing accepted Stage-1 artifact: {p}"
for name, path in OUTPUTS.items():
    if name not in {"prereg", "sample_manifest", "sampling_summary", "train_lowtp"}:
        assert not path.exists(), f"Scientific output already exists ({name}); stop recovery: {path}"

original_prereg = json.loads(OUTPUTS["prereg"].read_text(encoding="utf-8"))
assert original_prereg["experiment_id"] == EXPERIMENT_ID
assert int(original_prereg["region_size"]) == REGION_SIZE
assert int(original_prereg["epochs"]) == 4
assert abs(float(original_prereg["learning_rate"]) - 1e-3) < 1e-15
ORIGINAL_PREREG_SHA256 = hashlib.sha256(OUTPUTS["prereg"].read_bytes()).hexdigest()

AMENDMENT_PATH = RUN_DIR / "EXP046B_preregistration_amendment_001.json"
amendment = {
    "experiment_id": EXPERIMENT_ID,
    "run": "run_001",
    "amendment_id": "001",
    "timing": "after Stage-2 kernel restart; before any head training, train-contig CV, config selection, or validation evaluation",
    "reason": "restart-safe recovery and preregistered LR schedule correction before any scientific head result existed",
    "original_prereg_sha256": ORIGINAL_PREREG_SHA256,
    "scientific_design_unchanged": [
        "R32 absolute strand-specific grid",
        "fixed Stage-1 sample manifest and inverse-probability weights",
        "head A/B architecture and inputs",
        "class-balanced regional BCE",
        "alpha grid",
        "leave-one-train-contig-out CV selection",
        "single post-selection validation evaluation",
    ],
    "recovery_changes": {
        "cache_inference_batch_size": {"from": 2, "to": 1},
        "cache_mode": "existing memmaps opened r+; filled_u8 is completion journal",
        "flush_protocol": "feature memmaps flush first; completion bits set and flushed last",
        "legacy_filled_rows": "invalidate once and recompute because old writer set bits before durable flush",
        "pooling": "copy only required WIDTH512 36x32 slice to CPU; mean/max and score stats on CPU",
        "scaler_stat_chunk": STAT_CHUNK,
    },
    "training_schedule_change": {
        "from": "constant LR=1e-3 for 4 epochs",
        "to": TRAINING_SCHEDULE,
        "selection_based_on_seen_cv": False,
    },
    "validation_used_before_amendment": False,
}
if AMENDMENT_PATH.exists():
    existing_amendment = json.loads(AMENDMENT_PATH.read_text(encoding="utf-8"))
    assert existing_amendment == amendment, "Recovery amendment differs from frozen amendment on disk"
else:
    AMENDMENT_PATH.write_text(json.dumps(amendment, indent=2, ensure_ascii=False), encoding="utf-8")

print("Project:", PROJECT_ROOT)
print("Interpreter:", sys.executable)
print("AGENTS.md SHA256:", AGENTS_SHA256)
print("EXP046-B recovery run:", RUN_DIR)
print("Original prereg SHA256:", ORIGINAL_PREREG_SHA256)
print("Amendment:", AMENDMENT_PATH)
print("Validation/CV scientific outputs absent: PASS")


Project: D:\Projects\EPIC\EPIC
Interpreter: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe
AGENTS.md SHA256: bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3
EXP046-B recovery run: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32\run_001
Original prereg SHA256: c2c9575c383bac3dc4d39e667f42d6e78abe3b26aecda9a69ea118443a1eabea
Amendment: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32\run_001\EXP046B_preregistration_amendment_001.json
Validation/CV scientific outputs absent: PASS


In [2]:
# ============================================================
# LOAD DATA / FROZEN EXP045 / GEOMETRY SMOKE
# ============================================================
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

TRAIN_CONTIGS = sorted(set(split.intervals("train")["contig"].astype(str)))
VALIDATION_CONTIGS = sorted(set(split.intervals("validation")["contig"].astype(str)))
assert len(TRAIN_CONTIGS) == 9
assert len(VALIDATION_CONTIGS) == 3
assert set(TRAIN_CONTIGS).isdisjoint(VALIDATION_CONTIGS)
ALL_CONTIGS = TRAIN_CONTIGS + VALIDATION_CONTIGS
CONTIG_TO_CODE = {contig: i for i, contig in enumerate(ALL_CONTIGS)}

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)
assert profile_config.target_length == REGION_SIZE * REGIONS_PER_TILE
assert profile_config.context_flank >= REGION_SIZE * NEIGHBOR_REGIONS

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
model = create_cnn_presence_intensity(channels=512)
model.load_state_dict(checkpoint["model_state_dict"])
assert int(checkpoint["step"]) == 50_000
assert int(model.backbone.stem.out_channels) == 512
assert int(model.profile_head.in_channels) == 512
assert int(model.profile_head.out_channels) == 2
for p in model.parameters():
    p.requires_grad_(False)

device = torch_directml.device()
model = model.to(device)
model.eval()

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
train_tiles = train_generator.tile_index.copy().reset_index(drop=True)
required = {"contig", "target_start", "target_end", "allowed_position_strand"}
missing = required - set(train_tiles.columns)
assert not missing, ("train", missing)
assert ((train_tiles["target_end"] - train_tiles["target_start"]) == profile_config.target_length).all()
assert (train_tiles["target_start"].astype(np.int64) % REGION_SIZE == 0).all()
assert (train_tiles["target_end"].astype(np.int64) % REGION_SIZE == 0).all()

# EXP046-B deliberately does not instantiate/read the validation generator here.
# Validation inference and validation labels are first touched only after the CV-selected config is frozen.

# Verify x_both_strands ordering for B>1 against the established helper.
smoke_rows = train_tiles.iloc[:2]
smoke_batch = train_generator.make_batch(smoke_rows)
with torch.no_grad():
    smoke_x = smoke_batch.x_both_strands.to(device)
    smoke_feat = model.backbone(smoke_x)
    smoke_out = model.profile_head(smoke_feat)
    smoke_plus, smoke_minus_reverse, _, _ = forward_both_strands_multitask(
        model,
        smoke_x,
        batch_size=2,
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

c = profile_config.context_flank
L = profile_config.target_length
assert smoke_out.shape[0] == 4
assert torch.allclose(smoke_out[:2, 0:1, c:c+L].cpu(), smoke_plus.cpu(), atol=1e-6, rtol=1e-6)
assert torch.allclose(smoke_out[2:, 0:1, c:c+L].cpu(), smoke_minus_reverse.cpu(), atol=1e-6, rtol=1e-6)
assert smoke_feat.shape[1] == 512
assert smoke_feat.shape[-1] >= L + 2 * c

del smoke_x, smoke_feat, smoke_out, smoke_plus, smoke_minus_reverse, smoke_batch
gc.collect()

print("Frozen EXP045 geometry/order smoke: PASS")
print("Train tiles:", len(train_tiles))
print("Train positives:", int(train_generator.total_positive))
print("Validation generator not instantiated before CV selection: PASS")


Frozen EXP045 geometry/order smoke: PASS
Train tiles: 130525
Train positives: 170443
Validation generator not instantiated before CV selection: PASS


In [3]:
# ============================================================
# METRIC / REGION / FEATURE HELPERS
# ============================================================
def sigmoid_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x, dtype=np.float64)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    ex = np.exp(x[~positive])
    out[~positive] = ex / (1.0 + ex)
    return out.astype(np.float32)


def probability_bin_5dp_from_logit(logit):
    probability = sigmoid_numpy(logit)
    return np.rint(np.clip(probability, 0.0, 1.0) * 100_000).astype(np.uint32)


def update_histogram(hist, bins):
    bins = np.asarray(bins, dtype=np.int64)
    if bins.size:
        hist += np.bincount(bins, minlength=SUBMISSION_LEVELS).astype(np.int64)


def ap_from_histograms(positive_hist, total_hist):
    positives = int(positive_hist.sum())
    assert positives > 0
    pos_desc = positive_hist[::-1]
    total_desc = total_hist[::-1]
    tp = np.cumsum(pos_desc, dtype=np.int64)
    predicted = np.cumsum(total_desc, dtype=np.int64)
    precision = tp / np.maximum(predicted, 1)
    return float(np.sum((pos_desc / positives) * precision))


def exact_ap_from_scores(scores, labels):
    """Average precision on raw float32 scores with score ties grouped exactly."""
    scores = np.asarray(scores, dtype=np.float32)
    labels = np.asarray(labels, dtype=bool)
    assert scores.ndim == labels.ndim == 1
    assert len(scores) == len(labels)
    n_pos = int(labels.sum())
    assert n_pos > 0
    order = np.argsort(scores, kind="quicksort")[::-1]
    s = scores[order]
    y = labels[order]
    cumulative_tp = np.cumsum(y, dtype=np.int32)
    # End index of each distinct-score group. AP uses precision after the whole tie group.
    group_end = np.flatnonzero(np.r_[s[1:] != s[:-1], True])
    tp_end = cumulative_tp[group_end].astype(np.float64)
    tp_before = np.r_[0.0, tp_end[:-1]]
    positives_in_group = tp_end - tp_before
    precision_end = tp_end / (group_end.astype(np.float64) + 1.0)
    ap = float(np.sum((positives_in_group / n_pos) * precision_end))
    del order, s, y, cumulative_tp, group_end, tp_end, tp_before, positives_in_group, precision_end
    gc.collect()
    return ap


BIT_SHIFTS = np.arange(REGION_SIZE, dtype=np.uint32)

def bool_rows_to_u32(mask_2d):
    mask_2d = np.asarray(mask_2d, dtype=bool)
    assert mask_2d.ndim == 2 and mask_2d.shape[1] == REGION_SIZE
    weights = (np.uint32(1) << BIT_SHIFTS)[None, :]
    return np.sum(mask_2d.astype(np.uint32) * weights, axis=1, dtype=np.uint32)


def u32_to_bool_rows(values):
    values = np.asarray(values, dtype=np.uint32)
    return ((values[:, None] >> BIT_SHIFTS[None, :]) & np.uint32(1)).astype(bool)


KEY_SHIFT = np.uint64(56)
REGION_MASK = (np.uint64(1) << KEY_SHIFT) - np.uint64(1)

def pack_region_keys(contig_code, strand_code, region_ids):
    region_ids = np.asarray(region_ids, dtype=np.uint64)
    assert np.all(region_ids <= REGION_MASK)
    prefix = np.uint64(int(contig_code) * 2 + int(strand_code))
    return (prefix << KEY_SHIFT) | region_ids


def unpack_region_key(key):
    key = np.uint64(key)
    prefix = int(key >> KEY_SHIFT)
    return prefix // 2, prefix % 2, int(key & REGION_MASK)


def rank_strata_from_risk(keys, risks):
    """Return negative-stratum label per row, deterministic under score ties."""
    keys = np.asarray(keys, dtype=np.uint64)
    risks = np.asarray(risks, dtype=np.uint32)
    n = len(keys)
    assert n > 0
    order = np.lexsort((keys, -risks.astype(np.int64)))
    labels = np.empty(n, dtype=object)
    c1 = int(math.ceil(0.01 * n))
    c5 = int(math.ceil(0.05 * n))
    c15 = int(math.ceil(0.15 * n))
    labels[order[:c1]] = "0_1"
    labels[order[c1:c5]] = "1_5"
    labels[order[c5:c15]] = "5_15"
    labels[order[c15:]] = "15_100"
    return labels


def region_score_stats_torch(logit_regions):
    """logit_regions: [N_regions, 32] -> [4, N_regions]."""
    mean = logit_regions.mean(dim=-1)
    maximum = logit_regions.max(dim=-1).values
    std = torch.sqrt(torch.clamp(((logit_regions - mean[:, None]) ** 2).mean(dim=-1), min=0.0))
    peakiness = maximum - mean
    return torch.stack((mean, maximum, std, peakiness), dim=0)


def regionalize_extended_orientation(feature_ch_len, presence_logit_len, *, reverse_region_axis):
    """
    feature_ch_len: [512, full_length]
    presence_logit_len: [full_length]
    Returns score_stats [4, 36], pooled [1024, 36].

    We pool the 2 left-neighbor + 32 target + 2 right-neighbor absolute R32 regions.
    For the RC orientation, within-region reversal does not affect mean/max/std/peakiness;
    only the 36-region axis is flipped back to genomic left->right.
    """
    start = profile_config.context_flank - NEIGHBOR_REGIONS * REGION_SIZE
    length = profile_config.target_length + 2 * NEIGHBOR_REGIONS * REGION_SIZE
    assert length == 36 * REGION_SIZE
    stop = start + length
    assert start >= 0 and stop <= feature_ch_len.shape[-1]

    feat = feature_ch_len[:, start:stop].reshape(512, 36, REGION_SIZE)
    pooled_mean = feat.mean(dim=-1)
    pooled_max = feat.max(dim=-1).values
    pooled = torch.cat((pooled_mean, pooled_max), dim=0)  # [1024, 36]

    logits = presence_logit_len[start:stop].reshape(36, REGION_SIZE)
    stats = region_score_stats_torch(logits)  # [4, 36]

    if reverse_region_axis:
        pooled = torch.flip(pooled, dims=[-1])
        stats = torch.flip(stats, dims=[-1])
    return stats, pooled


def regionalize_extended_orientation_cpu(feature_ch_len_gpu, presence_logit_len_gpu, *, reverse_region_axis):
    """
    Restart-safe DirectML path. Copy only the required 36x32 slice to CPU, then
    compute mean/max and score statistics with NumPy. Returns float32 NumPy arrays:
    score_stats [4,36], pooled [1024,36].
    """
    start = profile_config.context_flank - NEIGHBOR_REGIONS * REGION_SIZE
    length = profile_config.target_length + 2 * NEIGHBOR_REGIONS * REGION_SIZE
    assert length == 36 * REGION_SIZE
    stop = start + length
    assert start >= 0 and stop <= feature_ch_len_gpu.shape[-1]

    feat_cpu = (
        feature_ch_len_gpu[:, start:stop]
        .detach().cpu().numpy().astype(np.float32, copy=True)
    )
    logit_cpu = (
        presence_logit_len_gpu[start:stop]
        .detach().cpu().numpy().astype(np.float32, copy=True)
    )

    feat = feat_cpu.reshape(512, 36, REGION_SIZE)
    pooled_mean = feat.mean(axis=-1, dtype=np.float32)
    pooled_max = feat.max(axis=-1)
    pooled = np.concatenate((pooled_mean, pooled_max), axis=0).astype(np.float32, copy=False)

    logits = logit_cpu.reshape(36, REGION_SIZE)
    mean = logits.mean(axis=-1, dtype=np.float32)
    maximum = logits.max(axis=-1)
    centered = logits - mean[:, None]
    std = np.sqrt(np.maximum((centered * centered).mean(axis=-1, dtype=np.float32), 0.0))
    peakiness = maximum - mean
    stats = np.stack((mean, maximum, std, peakiness), axis=0).astype(np.float32, copy=False)

    if reverse_region_axis:
        pooled = pooled[:, ::-1].copy()
        stats = stats[:, ::-1].copy()

    del feat_cpu, logit_cpu, feat, logits, centered, pooled_mean, pooled_max, mean, maximum, std, peakiness
    return stats, pooled


def genomic_target_base_logit(presence_logit_len, *, reverse_base_axis):
    base = presence_logit_len[c:c+L]
    if reverse_base_axis:
        base = torch.flip(base, dims=[-1])
    return base


def natural_prior_logit(active_n, total_n):
    pi = float(active_n) / float(total_n)
    assert 0.0 < pi < 1.0
    return float(math.log(pi / (1.0 - pi)))


def tie_aware_recall_at_budget(total_hist, cohort_bins, budget):
    """Expected cohort recall under random ordering inside the 5dp cutoff tie."""
    total_hist = np.asarray(total_hist, dtype=np.int64)
    cohort_bins = np.asarray(cohort_bins, dtype=np.int64)
    budget = int(min(max(budget, 0), total_hist.sum()))
    if len(cohort_bins) == 0:
        return float("nan")
    if budget == 0:
        return 0.0
    desc = total_hist[::-1]
    cumulative = np.cumsum(desc, dtype=np.int64)
    idx = int(np.searchsorted(cumulative, budget, side="left"))
    cutoff_bin = SUBMISSION_LEVELS - 1 - idx
    before = int(cumulative[idx - 1]) if idx > 0 else 0
    tie_total = int(total_hist[cutoff_bin])
    take_from_tie = budget - before
    tie_fraction = float(take_from_tie / tie_total) if tie_total else 0.0
    above = int((cohort_bins > cutoff_bin).sum())
    at = int((cohort_bins == cutoff_bin).sum())
    return float((above + tie_fraction * at) / len(cohort_bins))

print("Helpers loaded.")


Helpers loaded.


## Recovery Stage 1 — load and seal accepted train-only artifacts

Stage 1 is not recomputed. The notebook loads the accepted manifest, sampling summary, train LOW_TP cohort and population JSON, verifies exact structural invariants, and seals their current byte content with SHA256. On every later restart the same hashes must match.

The first recovery invocation deliberately invalidates the four legacy `filled_u8` bits from the crashed writer and recomputes those rows; subsequent restarts preserve the restart-safe journal.


In [4]:
# ============================================================
# RECOVERY STAGE 1 — LOAD / AUDIT / HASH-SEAL EXISTING ARTIFACTS
# ============================================================
def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

POPULATION_PATH = RUN_DIR / "train_region_population.json"
RECOVERY_INPUT_HASHES = RUN_DIR / "EXP046B_recovery_input_hashes.json"
RECOVERY_INIT_MARKER = RUN_DIR / "EXP046B_stage2_recovery_initialized.json"

sample_manifest = pd.read_csv(OUTPUTS["sample_manifest"])
sampling_summary = pd.read_csv(OUTPUTS["sampling_summary"])
train_lowtp = pd.read_csv(OUTPUTS["train_lowtp"])
population_payload = json.loads(POPULATION_PATH.read_text(encoding="utf-8"))
train_region_population = population_payload["per_contig"]

assert len(sample_manifest) == EXPECTED_TRAIN_SAMPLE_ROWS
assert int(sample_manifest["target"].sum()) == EXPECTED_TRAIN_SAMPLE_ACTIVE
assert int((sample_manifest["target"] == 0).sum()) == EXPECTED_TRAIN_SAMPLE_EMPTY
assert sample_manifest["region_key"].is_unique
assert sample_manifest["sample_row"].to_numpy(np.int64).tolist() == list(range(EXPECTED_TRAIN_SAMPLE_ROWS))
assert not sample_manifest.isna().any().any()

assert int(population_payload["total_valid_regions"]) == EXPECTED_TRAIN_VALID_REGIONS
assert int(population_payload["total_active_regions"]) == EXPECTED_TRAIN_ACTIVE_REGIONS
assert sum(int(v["empty_regions"]) for v in train_region_population.values()) == EXPECTED_TRAIN_EMPTY_REGIONS
assert set(train_region_population) == set(TRAIN_CONTIGS)

neg = sample_manifest[sample_manifest["target"] == 0].copy()
assert set(neg["negative_stratum"].astype(str)) == set(NEGATIVE_STRATA)
neg_counts = neg.groupby(["contig", "negative_stratum"], observed=True).size()
for contig in TRAIN_CONTIGS:
    assert int((neg["contig"].astype(str) == contig).sum()) == 10_000
    for stratum in NEGATIVE_STRATA:
        assert int(neg_counts.loc[(contig, stratum)]) == NEG_PER_STRATUM_PER_CONTIG

summary_neg = sampling_summary[sampling_summary["stratum"].isin(NEGATIVE_STRATA)].copy()
assert int(summary_neg["population_n"].sum()) == EXPECTED_TRAIN_EMPTY_REGIONS
assert int(summary_neg["sample_n"].sum()) == EXPECTED_TRAIN_SAMPLE_EMPTY
for row in summary_neg.itertuples(index=False):
    mask = (
        (neg["contig"].astype(str) == str(row.contig))
        & (neg["negative_stratum"].astype(str) == str(row.stratum))
    )
    d = neg.loc[mask]
    expected_p = float(row.sample_n) / float(row.population_n)
    assert len(d) == int(row.sample_n)
    assert np.allclose(d["inclusion_probability"].to_numpy(np.float64), expected_p, rtol=0, atol=1e-15)
    assert np.allclose(d["importance_weight"].to_numpy(np.float64), 1.0 / expected_p, rtol=0, atol=1e-12)

assert len(train_lowtp) == EXPECTED_TRAIN_LOWTP
assert train_lowtp[["contig", "strand", "position"]].duplicated().sum() == 0
assert not train_lowtp.isna().any().any()

seal_files = [
    OUTPUTS["sample_manifest"],
    OUTPUTS["sampling_summary"],
    OUTPUTS["train_lowtp"],
    POPULATION_PATH,
    OUTPUTS["prereg"],
    AMENDMENT_PATH,
]
current_seal = {
    p.name: {"size_bytes": int(p.stat().st_size), "sha256": sha256_file(p)}
    for p in seal_files
}
if RECOVERY_INPUT_HASHES.exists():
    sealed = json.loads(RECOVERY_INPUT_HASHES.read_text(encoding="utf-8"))
    assert sealed == current_seal, "Accepted recovery inputs changed since SHA256 seal"
else:
    RECOVERY_INPUT_HASHES.write_text(json.dumps(current_seal, indent=2), encoding="utf-8")

print("Stage 1 recovery audit: PASS")
print("  train regions:", f"{EXPECTED_TRAIN_VALID_REGIONS:,}")
print("  active regions:", f"{EXPECTED_TRAIN_ACTIVE_REGIONS:,}")
print("  fixed samples:", f"{len(sample_manifest):,}")
print("  train LOW_TP:", f"{len(train_lowtp):,}")
print("  input SHA256 seal:", RECOVERY_INPUT_HASHES)


Stage 1 recovery audit: PASS
  train regions: 7,284,784
  active regions: 83,887
  fixed samples: 173,887
  train LOW_TP: 42,613
  input SHA256 seal: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32\run_001\EXP046B_recovery_input_hashes.json


## Recovery Stage 2 — restart-safe WIDTH512 feature cache

The existing 1.67-GiB memmaps are **never reopened with `w+`**. Their exact byte sizes are checked, then they are opened with `r+`.

Durability protocol for each checkpoint:

1. write score/pool feature rows;
2. flush both feature memmaps;
3. set `filled_u8` only for those rows;
4. flush the journal.

Therefore a restart can only cause already-written feature rows to be recomputed; it cannot treat unflushed feature data as complete. DirectML only performs the frozen WIDTH512 forward pass. The required 36×32 feature slice is copied to CPU and regional mean/max pooling is computed there.


In [5]:
# ============================================================
# RECOVERY STAGE 2 — RESUME FIXED 5xR32 TRAIN SAMPLE FEATURE CACHE
# ============================================================
N_SAMPLES = len(sample_manifest)
score_context_path = CACHE_DIR / "score_context_f32.dat"
pool_context_path = CACHE_DIR / "backbone_meanmax_context_f16.dat"
filled_path = CACHE_DIR / "filled_u8.dat"
for p in (score_context_path, pool_context_path, filled_path):
    assert p.is_file(), f"Missing interrupted cache file: {p}"

expected_score_bytes = N_SAMPLES * SCORE_STAT_CHANNELS * CONTEXT_REGIONS * np.dtype(np.float32).itemsize
expected_pool_bytes = N_SAMPLES * BACKBONE_POOL_CHANNELS * CONTEXT_REGIONS * np.dtype(np.float16).itemsize
expected_filled_bytes = N_SAMPLES * np.dtype(np.uint8).itemsize
assert score_context_path.stat().st_size == expected_score_bytes, (score_context_path.stat().st_size, expected_score_bytes)
assert pool_context_path.stat().st_size == expected_pool_bytes, (pool_context_path.stat().st_size, expected_pool_bytes)
assert filled_path.stat().st_size == expected_filled_bytes, (filled_path.stat().st_size, expected_filled_bytes)

score_context_mm = np.memmap(
    score_context_path, mode="r+", dtype=np.float32,
    shape=(N_SAMPLES, SCORE_STAT_CHANNELS, CONTEXT_REGIONS),
)
pool_context_mm = np.memmap(
    pool_context_path, mode="r+", dtype=np.float16,
    shape=(N_SAMPLES, BACKBONE_POOL_CHANNELS, CONTEXT_REGIONS),
)
filled_mm = np.memmap(filled_path, mode="r+", dtype=np.uint8, shape=(N_SAMPLES,))
assert np.isin(np.asarray(filled_mm), [0, 1]).all(), "filled_u8 contains values other than 0/1"

# The original writer set completion bits before any durable data-first flush.
# On the first recovery invocation only, invalidate those legacy bits and recompute them.
if not RECOVERY_INIT_MARKER.exists():
    legacy_filled = np.flatnonzero(np.asarray(filled_mm) == 1).astype(np.int64)
    assert len(legacy_filled) <= 16, (
        "Unexpectedly many legacy completed rows. Recovery assumptions changed; inspect manually.",
        len(legacy_filled),
    )
    if len(legacy_filled):
        filled_mm[legacy_filled] = 0
        filled_mm.flush()
    recovery_init = {
        "legacy_filled_rows_invalidated": int(len(legacy_filled)),
        "cache_inference_batch_size": CACHE_INFERENCE_BATCH_SIZE,
        "flush_rows": CACHE_FLUSH_ROWS,
        "data_first_journal_last": True,
    }
    RECOVERY_INIT_MARKER.write_text(json.dumps(recovery_init, indent=2), encoding="utf-8")
    print("First recovery initialization: invalidated legacy rows =", len(legacy_filled))
else:
    recovery_init = json.loads(RECOVERY_INIT_MARKER.read_text(encoding="utf-8"))
    assert recovery_init["data_first_journal_last"] is True

key_to_sample_row = {
    int(k): int(r)
    for k, r in zip(
        sample_manifest["region_key"].to_numpy(np.uint64),
        sample_manifest["sample_row"].to_numpy(np.int64),
    )
}

pending_completion_rows = []
def durable_cache_checkpoint(force=False):
    global pending_completion_rows
    if not pending_completion_rows:
        return 0
    if (not force) and len(pending_completion_rows) < CACHE_FLUSH_ROWS:
        return 0
    unique_rows = np.unique(np.asarray(pending_completion_rows, dtype=np.int64))
    # Data first.
    score_context_mm.flush()
    pool_context_mm.flush()
    # Journal last.
    filled_mm[unique_rows] = 1
    filled_mm.flush()
    n = len(unique_rows)
    pending_completion_rows = []
    return n

started_stage2 = time.perf_counter()
initial_filled = int(np.asarray(filled_mm).sum())
print(f"Stage 2 resume: {initial_filled:,}/{N_SAMPLES:,} rows durable before inference")

with torch.no_grad():
    for contig_i, contig in enumerate(TRAIN_CONTIGS):
        contig_tiles = train_tiles[train_tiles["contig"].astype(str) == contig].reset_index(drop=True)
        contig_code = CONTIG_TO_CODE[contig]
        manifest_contig = sample_manifest[sample_manifest["contig"].astype(str) == contig]
        contig_rows = manifest_contig["sample_row"].to_numpy(np.int64)
        expected_contig = len(contig_rows)
        durable_before = int(np.asarray(filled_mm[contig_rows]).sum())
        inference_tiles = 0

        for tile_i in range(len(contig_tiles)):
            row = contig_tiles.iloc[tile_i]
            target_start = int(row["target_start"])
            first_region_id = target_start // REGION_SIZE
            target_region_ids = first_region_id + np.arange(REGIONS_PER_TILE, dtype=np.int64)

            needed_by_strand = []
            any_needed = False
            for strand_code in (0, 1):
                keys = pack_region_keys(contig_code, strand_code, target_region_ids)
                selected_pairs = []
                for j, key in enumerate(keys):
                    sample_row = key_to_sample_row.get(int(key))
                    if sample_row is not None and int(filled_mm[sample_row]) == 0:
                        selected_pairs.append((j, sample_row))
                needed_by_strand.append((keys, selected_pairs))
                any_needed = any_needed or bool(selected_pairs)

            if not any_needed:
                continue

            selected = contig_tiles.iloc[tile_i:tile_i + 1]
            batch = train_generator.make_batch(selected)
            assert len(selected) == CACHE_INFERENCE_BATCH_SIZE == 1
            x_gpu = batch.x_both_strands.to(device)
            feature_gpu = model.backbone(x_gpu)
            output_gpu = model.profile_head(feature_gpu)
            inference_tiles += 1

            for strand_code, orient_index, reverse_region_axis in (
                (0, 0, False),
                (1, 1, True),
            ):
                keys, selected_pairs = needed_by_strand[strand_code]
                if not selected_pairs:
                    continue

                stats, pooled = regionalize_extended_orientation_cpu(
                    feature_gpu[orient_index],
                    output_gpu[orient_index, 0],
                    reverse_region_axis=reverse_region_axis,
                )
                assert stats.shape == (SCORE_STAT_CHANNELS, 36)
                assert pooled.shape == (BACKBONE_POOL_CHANNELS, 36)
                assert np.isfinite(stats).all() and np.isfinite(pooled).all()

                for j, sample_row in selected_pairs:
                    # Center target region j has five-region context j:j+5 in the extended 36-region view.
                    score_context_mm[sample_row, :, :] = stats[:, j:j + CONTEXT_REGIONS]
                    pool_context_mm[sample_row, :, :] = pooled[:, j:j + CONTEXT_REGIONS].astype(np.float16)
                    pending_completion_rows.append(sample_row)

                del stats, pooled

            durable_cache_checkpoint(force=False)
            del x_gpu, feature_gpu, output_gpu, batch, selected

            if inference_tiles % 250 == 0:
                gc.collect()
                durable_now = int(np.asarray(filled_mm[contig_rows]).sum())
                print(
                    f"[{contig}] inference_tiles={inference_tiles:,} durable={durable_now:,}/{expected_contig:,} "
                    f"elapsed={(time.perf_counter()-started_stage2)/60:.1f} min"
                )

        durable_cache_checkpoint(force=True)
        gc.collect()
        durable_after = int(np.asarray(filled_mm[contig_rows]).sum())
        assert durable_after == expected_contig, (contig, durable_after, expected_contig)
        print(
            f"[{contig}] Stage-2 cache complete: {durable_after:,}/{expected_contig:,}; "
            f"new={durable_after-durable_before:,}; elapsed={(time.perf_counter()-started_stage2)/60:.1f} min"
        )

# Exact journal invariant before any CV.
durable_cache_checkpoint(force=True)
filled_count = int(np.asarray(filled_mm, dtype=np.uint8).sum())
assert filled_count == N_SAMPLES == EXPECTED_TRAIN_SAMPLE_ROWS, (filled_count, N_SAMPLES)
assert np.all(np.asarray(filled_mm) == 1)

# Full finite audit in small chunks; never materialize the 1.67-GiB pool cache in RAM.
AUDIT_CHUNK = 256
for s in range(0, N_SAMPLES, AUDIT_CHUNK):
    e = min(s + AUDIT_CHUNK, N_SAMPLES)
    score_chunk = np.asarray(score_context_mm[s:e], dtype=np.float32)
    pool_chunk = np.asarray(pool_context_mm[s:e], dtype=np.float32)
    assert np.isfinite(score_chunk).all(), ("nonfinite score cache", s, e)
    assert np.isfinite(pool_chunk).all(), ("nonfinite pool cache", s, e)
    del score_chunk, pool_chunk

stage2_complete = {
    "n_samples": int(N_SAMPLES),
    "filled_count": int(filled_count),
    "score_cache_bytes": int(score_context_path.stat().st_size),
    "pool_cache_bytes": int(pool_context_path.stat().st_size),
    "filled_cache_bytes": int(filled_path.stat().st_size),
    "cache_inference_batch_size": CACHE_INFERENCE_BATCH_SIZE,
    "pooling": "CPU after limited 36x32 WIDTH512 feature slice copy",
    "data_first_journal_last": True,
}
(RUN_DIR / "EXP046B_stage2_cache_complete.json").write_text(
    json.dumps(stage2_complete, indent=2), encoding="utf-8"
)
print("Stage 2 recovery complete: PASS")
print("Durable rows:", f"{filled_count:,}/{N_SAMPLES:,}")


First recovery initialization: invalidated legacy rows = 4
Stage 2 resume: 0/173,887 rows durable before inference
[NC_064035.1] inference_tiles=250 durable=524/25,280 elapsed=0.1 min
[NC_064035.1] inference_tiles=500 durable=1,173/25,280 elapsed=0.2 min
[NC_064035.1] inference_tiles=750 durable=1,694/25,280 elapsed=0.3 min
[NC_064035.1] inference_tiles=1,000 durable=2,213/25,280 elapsed=0.4 min
[NC_064035.1] inference_tiles=1,250 durable=2,862/25,280 elapsed=0.5 min
[NC_064035.1] inference_tiles=1,500 durable=3,636/25,280 elapsed=0.6 min
[NC_064035.1] inference_tiles=1,750 durable=4,159/25,280 elapsed=0.7 min
[NC_064035.1] inference_tiles=2,000 durable=4,811/25,280 elapsed=0.8 min
[NC_064035.1] inference_tiles=2,250 durable=5,335/25,280 elapsed=0.9 min
[NC_064035.1] inference_tiles=2,500 durable=5,987/25,280 elapsed=1.1 min
[NC_064035.1] inference_tiles=2,750 durable=6,632/25,280 elapsed=1.2 min
[NC_064035.1] inference_tiles=3,000 durable=7,145/25,280 elapsed=1.3 min
[NC_064035.1] inf

In [6]:
# ============================================================
# LOW-MEMORY PER-CONTIG CHANNEL SUFFICIENT STATISTICS
# Used to fit fold-local scalers without repeatedly scanning all samples.
# ============================================================
assert int(np.asarray(filled_mm).sum()) == N_SAMPLES
CHANNELS_B = VARIANT_CHANNELS["B"]
per_contig_sum = np.zeros((len(TRAIN_CONTIGS), CHANNELS_B), dtype=np.float64)
per_contig_sumsq = np.zeros((len(TRAIN_CONTIGS), CHANNELS_B), dtype=np.float64)
per_contig_count = np.zeros(len(TRAIN_CONTIGS), dtype=np.int64)
contig_to_train_index = {contig: i for i, contig in enumerate(TRAIN_CONTIGS)}

for contig in TRAIN_CONTIGS:
    ci = contig_to_train_index[contig]
    rows = sample_manifest.loc[sample_manifest["contig"].astype(str) == contig, "sample_row"].to_numpy(np.int64)
    for s in range(0, len(rows), STAT_CHUNK):
        idx = rows[s:s + STAT_CHUNK]
        score = np.asarray(score_context_mm[idx], dtype=np.float32)
        pool = np.asarray(pool_context_mm[idx], dtype=np.float32)

        # Reduce directly from float32 chunks into float64 accumulators.
        # No full float64 [chunk, 1024, 5] temporary is created.
        per_contig_sum[ci, :SCORE_STAT_CHANNELS] += score.sum(axis=(0, 2), dtype=np.float64)
        per_contig_sum[ci, SCORE_STAT_CHANNELS:] += pool.sum(axis=(0, 2), dtype=np.float64)
        per_contig_sumsq[ci, :SCORE_STAT_CHANNELS] += (score * score).sum(axis=(0, 2), dtype=np.float64)
        per_contig_sumsq[ci, SCORE_STAT_CHANNELS:] += (pool * pool).sum(axis=(0, 2), dtype=np.float64)
        per_contig_count[ci] += score.shape[0] * CONTEXT_REGIONS
        del score, pool

    print(f"[{contig}] scaler statistics complete: rows={len(rows):,}")
    gc.collect()

np.savez(
    OUTPUTS["channel_stats"],
    contigs=np.asarray(TRAIN_CONTIGS),
    channel_sum=per_contig_sum,
    channel_sumsq=per_contig_sumsq,
    channel_count=per_contig_count,
)


def scaler_for_training_contigs(training_contigs, variant):
    ids = [contig_to_train_index[c] for c in training_contigs]
    s = per_contig_sum[ids].sum(axis=0)
    ss = per_contig_sumsq[ids].sum(axis=0)
    n = int(per_contig_count[ids].sum())
    mean = s / n
    var = np.maximum(ss / n - mean * mean, 1e-8)
    std = np.sqrt(var)
    channels = VARIANT_CHANNELS[variant]
    mean = mean[:channels].astype(np.float32)
    std = std[:channels].astype(np.float32)
    assert np.isfinite(mean).all() and np.isfinite(std).all()
    assert np.all(std > 0)
    return mean, std

print("Per-contig low-memory scaler sufficient statistics saved: PASS")


[NC_064035.1] scaler statistics complete: rows=25,280
[NC_064036.1] scaler statistics complete: rows=22,955
[NC_064038.1] scaler statistics complete: rows=17,303
[NC_064039.1] scaler statistics complete: rows=21,172
[NC_064040.1] scaler statistics complete: rows=19,122
[NC_064043.1] scaler statistics complete: rows=19,934
[NC_064045.1] scaler statistics complete: rows=17,134
[NC_064047.1] scaler statistics complete: rows=15,683
[NC_064048.1] scaler statistics complete: rows=15,304
Per-contig low-memory scaler sufficient statistics saved: PASS


## Stage 3 — two frozen-head models under leave-one-train-contig-out CV

The expensive WIDTH512 backbone is never updated. A and B are trained from the same fixed sampled regions. The preregistration amendment freezes the training schedule **before any CV result**: 5% optimizer-step warm-up to 1e-3, then cosine decay to 1e-4 over the same four-epoch budget.

The head loss remains:

\[
L=0.5\,\overline{BCE}_{positive}+0.5\,\frac{\sum_{negative} w_i BCE_i}{\sum_{negative}w_i},
\qquad w_i=1/p_i.
\]

A fold-local channel scaler is fit from the eight training contigs only. The held-out train contig is never used for scaling, head fitting, class prior, or alpha fitting.


In [7]:
# ============================================================
# HEAD / TRAINING HELPERS — AMENDED 5% WARM-UP + COSINE
# ============================================================
class RegionalCorrectionHead(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, 64, kernel_size=1)
        self.act = nn.GELU()
        self.conv2 = nn.Conv1d(64, 1, kernel_size=5, padding=0)

    def forward(self, x):
        y = self.conv2(self.act(self.conv1(x)))
        assert y.shape[-1] == 1
        return y[:, 0, 0]


def make_optimizer(head):
    sig = inspect.signature(DirectMLAdam)
    if "weight_decay" in sig.parameters:
        return DirectMLAdam(head.parameters(), lr=HEAD_PEAK_LR, weight_decay=HEAD_WEIGHT_DECAY), False
    return DirectMLAdam(head.parameters(), lr=HEAD_PEAK_LR), True


def set_optimizer_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group["lr"] = float(lr)


def lr_for_optimizer_step(step_index, total_steps):
    """0-based optimizer step; exact peak and exact final LR are reached."""
    assert 0 <= step_index < total_steps
    warmup_steps = max(1, int(math.ceil(HEAD_WARMUP_FRACTION * total_steps)))
    if step_index < warmup_steps:
        return float(HEAD_PEAK_LR * (step_index + 1) / warmup_steps)
    decay_steps = total_steps - warmup_steps
    if decay_steps <= 1:
        return float(HEAD_FINAL_LR)
    decay_index = step_index - warmup_steps
    progress = decay_index / (decay_steps - 1)
    return float(
        HEAD_FINAL_LR
        + 0.5 * (HEAD_PEAK_LR - HEAD_FINAL_LR) * (1.0 + math.cos(math.pi * progress))
    )


def count_mixed_optimizer_steps(y_all):
    rng = np.random.default_rng(HEAD_SEED)
    total = 0
    for _epoch in range(HEAD_EPOCHS):
        order = rng.permutation(len(y_all))
        for bs in range(0, len(order), HEAD_BATCH_SIZE):
            loc = order[bs:bs + HEAD_BATCH_SIZE]
            y_np = y_all[loc]
            if len(y_np) and not np.all(y_np == y_np[0]):
                total += 1
    assert total > 0
    return total


def load_sample_batch(sample_rows, variant, mean, std):
    sample_rows = np.asarray(sample_rows, dtype=np.int64)
    score = np.asarray(score_context_mm[sample_rows], dtype=np.float32)
    if variant == "A":
        x = score
    else:
        pool = np.asarray(pool_context_mm[sample_rows], dtype=np.float32)
        x = np.concatenate((score, pool), axis=1)
    x = (x - mean[None, :, None]) / std[None, :, None]
    return torch.from_numpy(np.ascontiguousarray(x.astype(np.float32, copy=False)))


def balanced_ipw_bce(logit, target, importance_weight):
    per = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    pos = target > 0.5
    neg = ~pos
    assert bool(pos.any()) and bool(neg.any())
    pos_loss = per[pos].mean()
    neg_w = importance_weight[neg]
    neg_loss = (per[neg] * neg_w).sum() / torch.clamp(neg_w.sum(), min=1e-12)
    return 0.5 * (pos_loss + neg_loss)


def train_head(training_contigs, variant, fold_name):
    training_contigs = list(training_contigs)
    rows = sample_manifest.loc[sample_manifest["contig"].isin(training_contigs), "sample_row"].to_numpy(np.int64)
    y_all = sample_manifest.loc[rows, "target"].to_numpy(np.float32)
    w_all = sample_manifest.loc[rows, "importance_weight"].to_numpy(np.float32)
    assert int((y_all == 1).sum()) > 0 and int((y_all == 0).sum()) > 0

    mean, std = scaler_for_training_contigs(training_contigs, variant)
    torch.manual_seed(HEAD_SEED)
    head = RegionalCorrectionHead(VARIANT_CHANNELS[variant]).to(device)
    optimizer, manual_wd = make_optimizer(head)
    total_optimizer_steps = count_mixed_optimizer_steps(y_all)
    warmup_steps = max(1, int(math.ceil(HEAD_WARMUP_FRACTION * total_optimizer_steps)))
    rng = np.random.default_rng(HEAD_SEED)
    history = []
    optimizer_step = 0

    head.train()
    for epoch in range(1, HEAD_EPOCHS + 1):
        order = rng.permutation(len(rows))
        epoch_loss_num = 0.0
        epoch_batches = 0
        epoch_first_lr = None
        epoch_last_lr = None

        for bs in range(0, len(order), HEAD_BATCH_SIZE):
            loc = order[bs:bs + HEAD_BATCH_SIZE]
            y_np = y_all[loc]
            if np.all(y_np == y_np[0]):
                continue

            current_lr = lr_for_optimizer_step(optimizer_step, total_optimizer_steps)
            set_optimizer_lr(optimizer, current_lr)
            if epoch_first_lr is None:
                epoch_first_lr = current_lr
            epoch_last_lr = current_lr

            sample_rows = rows[loc]
            x = load_sample_batch(sample_rows, variant, mean, std).to(device)
            y = torch.from_numpy(y_np).to(device)
            iw = torch.from_numpy(w_all[loc]).to(device)

            optimizer.zero_grad(set_to_none=True)
            logit = head(x)
            loss = balanced_ipw_bce(logit, y, iw)
            loss.backward()
            optimizer.step()
            if manual_wd and HEAD_WEIGHT_DECAY > 0:
                with torch.no_grad():
                    decay = 1.0 - current_lr * HEAD_WEIGHT_DECAY
                    for p in head.parameters():
                        p.mul_(decay)

            epoch_loss_num += float(loss.detach().cpu().item())
            epoch_batches += 1
            optimizer_step += 1
            del x, y, iw, logit, loss

        mean_loss = epoch_loss_num / max(epoch_batches, 1)
        history.append({
            "fold": fold_name,
            "variant": variant,
            "epoch": epoch,
            "loss": mean_loss,
            "optimizer_steps_completed": optimizer_step,
            "optimizer_steps_total": total_optimizer_steps,
            "warmup_steps": warmup_steps,
            "epoch_first_lr": epoch_first_lr,
            "epoch_last_lr": epoch_last_lr,
        })
        print(
            f"[{fold_name}][{variant}] epoch {epoch}/{HEAD_EPOCHS} loss={mean_loss:.6f} "
            f"lr={epoch_first_lr:.3g}->{epoch_last_lr:.3g} step={optimizer_step}/{total_optimizer_steps}"
        )

    assert optimizer_step == total_optimizer_steps
    assert abs(lr_for_optimizer_step(total_optimizer_steps - 1, total_optimizer_steps) - HEAD_FINAL_LR) <= 1e-15

    active_n = sum(train_region_population[c]["active_regions"] for c in training_contigs)
    total_n = sum(train_region_population[c]["valid_regions"] for c in training_contigs)
    prior_offset = natural_prior_logit(active_n, total_n)

    head.eval()
    return head, mean, std, prior_offset, pd.DataFrame(history)


def head_region_logits_from_extended(stats_4x36, pooled_1024x36, head, variant, mean, std, prior_offset):
    if variant == "A":
        x_np = stats_4x36
    else:
        pool_q = pooled_1024x36.astype(np.float16).astype(np.float32)
        x_np = np.concatenate((stats_4x36, pool_q), axis=0)
    x_np = (x_np - mean[:, None]) / std[:, None]
    x = torch.from_numpy(np.ascontiguousarray(x_np[None].astype(np.float32))).to(device)
    with torch.no_grad():
        raw = head.conv2(head.act(head.conv1(x)))[:, 0, :]
    g = raw[0].detach().cpu().numpy().astype(np.float32, copy=True) + np.float32(prior_offset)
    del x, raw
    assert g.shape == (REGIONS_PER_TILE,)
    return g

# Scheduler shape smoke test — does not use data.
for _n in (20, 101, 1000):
    _lr = np.asarray([lr_for_optimizer_step(i, _n) for i in range(_n)], dtype=np.float64)
    _w = max(1, int(math.ceil(HEAD_WARMUP_FRACTION * _n)))
    assert np.isfinite(_lr).all()
    assert abs(_lr[-1] - HEAD_FINAL_LR) <= 1e-15
    assert _lr.max() <= HEAD_PEAK_LR + 1e-15
    assert np.all(np.diff(_lr[_w - 1:]) <= 1e-15)

print("Head/training helpers loaded with frozen amendment schedule:", TRAINING_SCHEDULE)


Head/training helpers loaded with frozen amendment schedule: {'type': 'optimizer_step_warmup_plus_cosine', 'epochs': 4, 'warmup_fraction': 0.05, 'peak_lr': 0.001, 'final_lr': 0.0001}


In [8]:
# ============================================================
# CV EVALUATION HELPERS
# ============================================================
def empty_danger_strata(keys, risks, active):
    keys = np.asarray(keys, dtype=np.uint64)
    risks = np.asarray(risks, dtype=np.uint32)
    active = np.asarray(active, dtype=bool)
    labels = np.full(len(keys), "ACTIVE", dtype=object)
    empty_idx = np.flatnonzero(~active)
    labels[empty_idx] = rank_strata_from_risk(keys[empty_idx], risks[empty_idx])
    return labels


def regional_diagnostics(contig, variant, region_g, region_active, region_risk, region_key, phase):
    region_g = np.asarray(region_g, dtype=np.float32)
    region_active = np.asarray(region_active, dtype=bool)
    region_risk = np.asarray(region_risk, dtype=np.uint32)
    region_key = np.asarray(region_key, dtype=np.uint64)
    strata = empty_danger_strata(region_key, region_risk, region_active)
    rows = []
    auc = float(roc_auc_score(region_active.astype(np.uint8), region_g))

    active_g = np.sort(region_g[region_active])
    for recall_target in REGIONAL_RECALL_DIAGNOSTIC_TARGETS:
        # Suppress low-g regions. Choose threshold from active scores so at least target fraction is retained.
        drop_active_n = int(math.floor((1.0 - recall_target) * len(active_g)))
        if drop_active_n <= 0:
            threshold = -np.inf
        else:
            threshold = float(active_g[min(drop_active_n - 1, len(active_g) - 1)])
        suppressed = region_g <= threshold
        realized_active_recall = float((~suppressed[region_active]).mean())
        row = {
            "phase": phase,
            "contig": contig,
            "variant": variant,
            "regional_auc": auc,
            "target_active_recall": recall_target,
            "realized_active_recall": realized_active_recall,
            "suppressed_empty_all": float(suppressed[~region_active].mean()),
        }
        for stratum in NEGATIVE_STRATA:
            m = strata == stratum
            row[f"suppressed_empty_{stratum}"] = float(suppressed[m].mean()) if int(m.sum()) else float("nan")
        rows.append(row)
    return rows


def evaluate_cv_fold_contig(contig, head_A, mean_A, std_A, prior_A, head_B, mean_B, std_B, prior_B, raw_cache_path):
    contig_tiles = train_tiles[train_tiles["contig"].astype(str) == contig].reset_index(drop=True)
    contig_code = CONTIG_TO_CODE[contig]

    hist_total = {(v, a): np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for v in ("A", "B") for a in ALPHAS}
    hist_pos = {(v, a): np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for v in ("A", "B") for a in ALPHAS}

    lowtp_contig = train_lowtp[train_lowtp["contig"].astype(str) == contig].copy()
    lowtp_bins = {(v, a): [] for v in ("A", "B") for a in ALPHAS}

    region_keys = []
    region_risks = []
    region_active = []
    region_g_A = []
    region_g_B = []
    raw_base_regions = []
    raw_valid_masks = []
    raw_positive_masks = []

    positions_seen = 0
    positives_seen = 0
    with torch.no_grad():
        for start_i in range(0, len(contig_tiles), EVAL_INFERENCE_BATCH_SIZE):
            selected = contig_tiles.iloc[start_i:start_i + EVAL_INFERENCE_BATCH_SIZE]
            batch = train_generator.make_batch(selected)
            B = len(selected)
            assert B == 1, "CV evaluator currently preregistered with batch size 1 for unambiguous audit"
            x_gpu = batch.x_both_strands.to(device)
            feature_gpu = model.backbone(x_gpu)
            output_gpu = model.profile_head(feature_gpu)

            row = selected.iloc[0]
            target_start = int(row["target_start"])
            first_region_id = target_start // REGION_SIZE
            target_region_ids = first_region_id + np.arange(REGIONS_PER_TILE, dtype=np.int64)

            plus_count = batch.count[0, 0, :].numpy().astype(np.float32, copy=False)
            minus_count = batch.count[0, 1, :].numpy().astype(np.float32, copy=False)
            plus_mask = batch.mask[0, 0, :].numpy().astype(bool, copy=False)
            minus_mask = batch.mask[0, 1, :].numpy().astype(bool, copy=False)

            for strand_code, strand_name, orient_index, reverse_axis, count, mask in (
                (0, "+", 0, False, plus_count, plus_mask),
                (1, "-", 1, True, minus_count, minus_mask),
            ):
                stats, pool = regionalize_extended_orientation_cpu(
                    feature_gpu[orient_index], output_gpu[orient_index, 0], reverse_region_axis=reverse_axis
                )
                base_logit = genomic_target_base_logit(
                    output_gpu[orient_index, 0], reverse_base_axis=reverse_axis
                ).detach().cpu().numpy().astype(np.float32, copy=True)

                g_A = head_region_logits_from_extended(stats, pool, head_A, "A", mean_A, std_A, prior_A)
                g_B = head_region_logits_from_extended(stats, pool, head_B, "B", mean_B, std_B, prior_B)

                valid_2d = mask.reshape(REGIONS_PER_TILE, REGION_SIZE)
                positive_2d = ((count > 0) & mask).reshape(REGIONS_PER_TILE, REGION_SIZE)
                valid_regions = valid_2d.any(axis=1)
                base_2d = base_logit.reshape(REGIONS_PER_TILE, REGION_SIZE)
                base_bins_2d = probability_bin_5dp_from_logit(base_2d)
                risk = np.where(valid_2d, base_bins_2d, np.uint32(0)).max(axis=1).astype(np.uint32)
                active = positive_2d.any(axis=1)
                keys = pack_region_keys(contig_code, strand_code, target_region_ids)

                region_keys.append(keys[valid_regions])
                region_risks.append(risk[valid_regions])
                region_active.append(active[valid_regions])
                region_g_A.append(g_A[valid_regions])
                region_g_B.append(g_B[valid_regions])

                raw_base_regions.append(base_2d[valid_regions].copy())
                raw_valid_masks.append(bool_rows_to_u32(valid_2d[valid_regions]))
                raw_positive_masks.append(bool_rows_to_u32(positive_2d[valid_regions]))

                valid = mask
                positive = (count > 0) & mask
                positions_seen += int(valid.sum())
                positives_seen += int(positive.sum())

                for variant, g in (("A", g_A), ("B", g_B)):
                    g_base = np.repeat(g, REGION_SIZE)
                    for alpha in ALPHAS:
                        final_logit = base_logit + np.float32(alpha) * g_base
                        bins = probability_bin_5dp_from_logit(final_logit)
                        update_histogram(hist_total[(variant, alpha)], bins[valid])
                        update_histogram(hist_pos[(variant, alpha)], bins[positive])

                        # Fixed train LOW_TP coordinate scores for same-budget audit.
                        sub = lowtp_contig[
                            (lowtp_contig["strand"] == strand_name)
                            & (lowtp_contig["position"] >= target_start)
                            & (lowtp_contig["position"] < target_start + L)
                        ]
                        if len(sub):
                            local = (sub["position"].to_numpy(np.int64) - target_start).astype(np.int64)
                            lowtp_bins[(variant, alpha)].extend(bins[local].astype(np.int64).tolist())

            del x_gpu, feature_gpu, output_gpu, batch, selected

    assert positions_seen == int(contig_tiles["allowed_position_strand"].sum())
    assert positives_seen > 0
    assert all(len(lowtp_bins[(v, a)]) == len(lowtp_contig) for v in ("A", "B") for a in ALPHAS)

    # Save compact raw fold cache once: base logits per R32 + two region corrections + bit masks.
    key_arr = np.concatenate(region_keys).astype(np.uint64, copy=False)
    risk_arr = np.concatenate(region_risks).astype(np.uint32, copy=False)
    active_arr = np.concatenate(region_active).astype(bool, copy=False)
    gA_arr = np.concatenate(region_g_A).astype(np.float32, copy=False)
    gB_arr = np.concatenate(region_g_B).astype(np.float32, copy=False)
    base_arr = np.concatenate(raw_base_regions).astype(np.float32, copy=False)
    valid_mask_arr = np.concatenate(raw_valid_masks).astype(np.uint32, copy=False)
    pos_mask_arr = np.concatenate(raw_positive_masks).astype(np.uint32, copy=False)
    assert len(np.unique(key_arr)) == len(key_arr)
    np.savez(
        raw_cache_path,
        region_key=key_arr,
        region_risk5=risk_arr,
        region_active=active_arr.astype(np.uint8),
        g_A=gA_arr,
        g_B=gB_arr,
        base_logit=base_arr,
        valid_mask=valid_mask_arr,
        positive_mask=pos_mask_arr,
    )

    fold_metrics = []
    for variant in ("A", "B"):
        for alpha in ALPHAS:
            ap5 = ap_from_histograms(hist_pos[(variant, alpha)], hist_total[(variant, alpha)])
            fold_metrics.append({
                "heldout_contig": contig,
                "variant": variant,
                "alpha": alpha,
                "ap5": ap5,
                "positive_n": int(hist_pos[(variant, alpha)].sum()),
                "position_n": int(hist_total[(variant, alpha)].sum()),
                "lowtp_n": len(lowtp_bins[(variant, alpha)]),
            })

    region_diag = []
    region_diag += regional_diagnostics(contig, "A", gA_arr, active_arr, risk_arr, key_arr, "train_cv")
    region_diag += regional_diagnostics(contig, "B", gB_arr, active_arr, risk_arr, key_arr, "train_cv")

    return fold_metrics, region_diag, hist_total, hist_pos, lowtp_bins

print("CV evaluation helpers loaded.")


CV evaluation helpers loaded.


In [ ]:
# ============================================================
# RUN 9-FOLD TRAIN-CONTIG CV — TRAIN A+B, EVALUATE BOTH IN ONE BACKBONE PASS
# ============================================================
CV_RAW_DIR.mkdir(parents=False, exist_ok=False)
all_fold_metrics = []
all_region_diag = []
all_history = []
pooled_total_hist = {(v, a): np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for v in ("A", "B") for a in ALPHAS}
pooled_pos_hist = {(v, a): np.zeros(SUBMISSION_LEVELS, dtype=np.int64) for v in ("A", "B") for a in ALPHAS}
pooled_lowtp_bins = {(v, a): [] for v in ("A", "B") for a in ALPHAS}

cv_started = time.perf_counter()
for fold_i, heldout in enumerate(TRAIN_CONTIGS, 1):
    training_contigs = [c for c in TRAIN_CONTIGS if c != heldout]
    print("\n" + "=" * 88)
    print(f"FOLD {fold_i}/9 — held out {heldout}")
    print("=" * 88)

    head_A, mean_A, std_A, prior_A, hist_A = train_head(training_contigs, "A", heldout)
    head_B, mean_B, std_B, prior_B, hist_B = train_head(training_contigs, "B", heldout)
    all_history.extend([hist_A, hist_B])

    torch.save({
        "heldout_contig": heldout,
        "variant": "A",
        "state_dict": {k: v.detach().cpu() for k, v in head_A.state_dict().items()},
        "scaler_mean": mean_A,
        "scaler_std": std_A,
        "prior_offset": prior_A,
        "training_schedule": TRAINING_SCHEDULE,
    }, FOLD_MODEL_DIR / f"fold_{fold_i:02d}_{heldout}_A.pt")
    torch.save({
        "heldout_contig": heldout,
        "variant": "B",
        "state_dict": {k: v.detach().cpu() for k, v in head_B.state_dict().items()},
        "scaler_mean": mean_B,
        "scaler_std": std_B,
        "prior_offset": prior_B,
        "training_schedule": TRAINING_SCHEDULE,
    }, FOLD_MODEL_DIR / f"fold_{fold_i:02d}_{heldout}_B.pt")

    fold_metrics, region_diag, fold_total, fold_pos, fold_lowtp = evaluate_cv_fold_contig(
        heldout,
        head_A, mean_A, std_A, prior_A,
        head_B, mean_B, std_B, prior_B,
        CV_RAW_DIR / f"{heldout}_raw_region_cache.npz",
    )
    all_fold_metrics.extend(fold_metrics)
    all_region_diag.extend(region_diag)
    for key in pooled_total_hist:
        pooled_total_hist[key] += fold_total[key]
        pooled_pos_hist[key] += fold_pos[key]
        pooled_lowtp_bins[key].extend(fold_lowtp[key])

    del head_A, head_B
    gc.collect()
    print(f"Fold {fold_i} complete; CV elapsed={(time.perf_counter()-cv_started)/60:.1f} min")

cv_fold_df = pd.DataFrame(all_fold_metrics)
cv_region_diag_df = pd.DataFrame(all_region_diag)
training_history_df = pd.concat(all_history, ignore_index=True)
training_history_df.to_csv(RUN_DIR / "head_training_history.csv", index=False)
cv_fold_df.to_csv(OUTPUTS["cv_fold_ap"], index=False)
cv_region_diag_df.to_csv(OUTPUTS["cv_region_diag"], index=False)

# Alpha=0 must be exactly independent of head variant, fold by fold.
for contig in TRAIN_CONTIGS:
    a0 = cv_fold_df[(cv_fold_df.heldout_contig == contig) & (cv_fold_df.variant == "A") & (cv_fold_df.alpha == 0.0)].iloc[0].ap5
    b0 = cv_fold_df[(cv_fold_df.heldout_contig == contig) & (cv_fold_df.variant == "B") & (cv_fold_df.alpha == 0.0)].iloc[0].ap5
    assert a0 == b0, (contig, a0, b0)

pooled_rows = []
for variant in ("A", "B"):
    for alpha in ALPHAS:
        pooled_rows.append({
            "variant": variant,
            "alpha": alpha,
            "pooled_oof_ap5": ap_from_histograms(pooled_pos_hist[(variant, alpha)], pooled_total_hist[(variant, alpha)]),
            "position_n": int(pooled_total_hist[(variant, alpha)].sum()),
            "positive_n": int(pooled_pos_hist[(variant, alpha)].sum()),
            "mean_fold_ap5": float(cv_fold_df[(cv_fold_df.variant == variant) & (cv_fold_df.alpha == alpha)].ap5.mean()),
            "min_fold_ap5": float(cv_fold_df[(cv_fold_df.variant == variant) & (cv_fold_df.alpha == alpha)].ap5.min()),
            "max_fold_ap5": float(cv_fold_df[(cv_fold_df.variant == variant) & (cv_fold_df.alpha == alpha)].ap5.max()),
        })
cv_pooled_df = pd.DataFrame(pooled_rows)
cv_pooled_df.to_csv(OUTPUTS["cv_pooled_ap"], index=False)

# Same-budget positive and fixed train-LOW_TP recall on pooled train-contig OOF predictions.
cv_budget_rows = []
for variant in ("A", "B"):
    for alpha in ALPHAS:
        total_hist = pooled_total_hist[(variant, alpha)]
        pos_hist = pooled_pos_hist[(variant, alpha)]
        n_pos = int(pos_hist.sum())
        positive_bins = np.repeat(np.arange(SUBMISSION_LEVELS, dtype=np.int64), pos_hist)
        low_bins = np.asarray(pooled_lowtp_bins[(variant, alpha)], dtype=np.int64)
        for multiplier in BUDGET_MULTIPLIERS:
            budget = multiplier * n_pos
            cv_budget_rows.append({
                "variant": variant,
                "alpha": alpha,
                "budget_multiplier": multiplier,
                "budget_positions": int(budget),
                "positive_position_recall": tie_aware_recall_at_budget(total_hist, positive_bins, budget),
                "fixed_train_lowtp_recall": tie_aware_recall_at_budget(total_hist, low_bins, budget),
            })
cv_budget_df = pd.DataFrame(cv_budget_rows)
cv_budget_df.to_csv(OUTPUTS["cv_budget"], index=False)

print("\nPooled OOF AP5:")
display(cv_pooled_df.sort_values(["pooled_oof_ap5", "alpha"], ascending=[False, True]))



FOLD 1/9 — held out NC_064035.1


C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\Lib\site-packages\torch\nn\functional.py:3244: UserWarning: The operator 'aten::log_sigmoid_forward' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  return torch.binary_cross_entropy_with_logits(input, target, weight, pos_weight, reduction_enum)


[NC_064035.1][A] epoch 1/4 loss=0.420601 lr=8.55e-06->0.000906 step=581/2324
[NC_064035.1][A] epoch 2/4 loss=0.370561 lr=0.000905->0.000588 step=1162/2324
[NC_064035.1][A] epoch 3/4 loss=0.370386 lr=0.000587->0.000245 step=1743/2324
[NC_064035.1][A] epoch 4/4 loss=0.369682 lr=0.000245->0.0001 step=2324/2324
[NC_064035.1][B] epoch 1/4 loss=0.400438 lr=8.55e-06->0.000906 step=581/2324
[NC_064035.1][B] epoch 2/4 loss=0.366142 lr=0.000905->0.000588 step=1162/2324
[NC_064035.1][B] epoch 3/4 loss=0.354461 lr=0.000587->0.000245 step=1743/2324
[NC_064035.1][B] epoch 4/4 loss=0.343045 lr=0.000245->0.0001 step=2324/2324


In [ ]:
# ============================================================
# FREEZE ONE CONFIGURATION FROM TRAIN-CONTIG CV — BEFORE VALIDATION
# ============================================================
selection_table = cv_pooled_df.sort_values(
    ["pooled_oof_ap5", "alpha", "variant"],
    ascending=[False, True, True],
    kind="mergesort",
).reset_index(drop=True)
selected = selection_table.iloc[0]
SELECTED_VARIANT = str(selected["variant"])
SELECTED_ALPHA = float(selected["alpha"])
SELECTED_CV_AP5 = float(selected["pooled_oof_ap5"])

# Control comparison: best A vs best B on the same pooled train-CV protocol.
best_A = cv_pooled_df[cv_pooled_df.variant == "A"].sort_values(["pooled_oof_ap5", "alpha"], ascending=[False, True]).iloc[0]
best_B = cv_pooled_df[cv_pooled_df.variant == "B"].sort_values(["pooled_oof_ap5", "alpha"], ascending=[False, True]).iloc[0]

selected_payload = {
    "selected_variant": SELECTED_VARIANT,
    "selected_alpha": SELECTED_ALPHA,
    "selected_pooled_train_cv_ap5": SELECTED_CV_AP5,
    "best_A": {"alpha": float(best_A.alpha), "pooled_oof_ap5": float(best_A.pooled_oof_ap5)},
    "best_B": {"alpha": float(best_B.alpha), "pooled_oof_ap5": float(best_B.pooled_oof_ap5)},
    "B_minus_A_best_ap5": float(best_B.pooled_oof_ap5 - best_A.pooled_oof_ap5),
    "selection_rule": "max pooled train-contig OOF AP5; then smaller alpha; then A lexical tie-break",
    "exp046b_validation_inference_or_labels_used_for_selection": False,
}
OUTPUTS["selected_config"].write_text(
    json.dumps(selected_payload, indent=2, ensure_ascii=False), encoding="utf-8"
)

# CV alpha-curve plot.
fig, ax = plt.subplots(figsize=(8, 5))
for variant in ("A", "B"):
    d = cv_pooled_df[cv_pooled_df.variant == variant].sort_values("alpha")
    ax.plot(d.alpha, d.pooled_oof_ap5, marker="o", label=f"Variant {variant}")
ax.set_xlabel("alpha")
ax.set_ylabel("pooled train-contig OOF nucleotide AP5")
ax.set_title("EXP046-B train-contig CV: regional correction strength")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUTS["cv_plot"], dpi=160)
plt.close(fig)

print("CONFIG FROZEN BEFORE VALIDATION")
print(json.dumps(selected_payload, indent=2))
assert OUTPUTS["selected_config"].is_file()


## Raw-score audit after CV selection

Configuration selection above used only the pre-registered **5-decimal natural nucleotide AP** over pooled train-contig OOF predictions.

To ensure a gain is not merely a tie-breaking artifact, the compact per-fold raw caches now compute **unrounded float32-logit AP** for:

- EXP045 (`alpha=0`), and
- the one globally selected `(variant, alpha)`.

The audit is reported per held-out train contig. We intentionally do not sort all nine train contigs jointly in RAM because that would spend substantial memory on a diagnostic that does not participate in model selection.


In [ ]:
# ============================================================
# RAW AP AUDIT — SELECTED CONFIG ONLY, PER HELD-OUT TRAIN CONTIG
# ============================================================
raw_audit_rows = []
for contig in TRAIN_CONTIGS:
    data = np.load(CV_RAW_DIR / f"{contig}_raw_region_cache.npz", allow_pickle=False)
    base = data["base_logit"].astype(np.float32, copy=False)
    valid = u32_to_bool_rows(data["valid_mask"])
    positive = u32_to_bool_rows(data["positive_mask"])
    g = data[f"g_{SELECTED_VARIANT}"].astype(np.float32, copy=False)

    base_scores = base[valid]
    labels = positive[valid]
    selected_scores = (base + np.float32(SELECTED_ALPHA) * g[:, None])[valid]
    baseline_raw_ap = exact_ap_from_scores(base_scores, labels)
    selected_raw_ap = exact_ap_from_scores(selected_scores, labels)
    raw_audit_rows.append({
        "heldout_contig": contig,
        "baseline_raw_ap": baseline_raw_ap,
        "selected_raw_ap": selected_raw_ap,
        "delta_raw_ap": selected_raw_ap - baseline_raw_ap,
        "selected_variant": SELECTED_VARIANT,
        "selected_alpha": SELECTED_ALPHA,
    })
    print(contig, "raw AP", baseline_raw_ap, "->", selected_raw_ap)
    del data, base, valid, positive, g, base_scores, labels, selected_scores
    gc.collect()

cv_raw_audit_df = pd.DataFrame(raw_audit_rows)
cv_raw_audit_df.to_csv(OUTPUTS["cv_raw_audit"], index=False)
display(cv_raw_audit_df)


## Stage 4 — fit one final head on all nine train contigs

Only after `(variant, alpha)` is frozen from train-contig CV, the selected head is retrained once on all nine train contigs using the same architecture, same sample manifest, same loss and the **amended four-epoch 5% warm-up + cosine schedule**. No validation information is used in this fit.


In [ ]:
# ============================================================
# FINAL TRAIN-ONLY HEAD FIT
# ============================================================
final_head, final_mean, final_std, final_prior_offset, final_history = train_head(
    TRAIN_CONTIGS, SELECTED_VARIANT, "ALL_TRAIN"
)
final_history.to_csv(RUN_DIR / "final_head_training_history.csv", index=False)

torch.save({
    "experiment_id": EXPERIMENT_ID,
    "variant": SELECTED_VARIANT,
    "alpha": SELECTED_ALPHA,
    "state_dict": {k: v.detach().cpu() for k, v in final_head.state_dict().items()},
    "scaler_mean": final_mean,
    "scaler_std": final_std,
    "prior_offset": final_prior_offset,
    "channels": VARIANT_CHANNELS[SELECTED_VARIANT],
    "region_size": REGION_SIZE,
    "epochs": HEAD_EPOCHS,
    "training_schedule": TRAINING_SCHEDULE,
    "weight_decay": HEAD_WEIGHT_DECAY,
    "checkpoint_source": str(CHECKPOINT_PATH),
}, OUTPUTS["final_head"])

print("Final head trained on all 9 train contigs.")
print("Validation has not yet been evaluated in this notebook.")


## Stage 5 — single final evaluation on the three untouched validation contigs

This stage is the only learned-model use of validation labels in EXP046-B.

It records:

- overall and per-contig AP5 for baseline and selected correction;
- raw (unrounded) AP overall and per contig;
- positive-position and fixed LOW_TP recall at identical top-K budgets (`1x, 2x, 5x, 10x` the number of positives);
- diagnostics over the **whole danger-ranked empty tail** at fixed active-region recall targets;
- score movement for the already-fixed validation LOW_TP and HARD_FP coordinate cohorts.

No hyperparameter or sampling decision may be changed after this stage begins.


In [ ]:
# ============================================================
# FINAL VALIDATION — BASELINE + ONE FROZEN CONFIG ONLY
# ============================================================
assert OUTPUTS["selected_config"].is_file()

# First EXP046-B access to validation inference/labels occurs here, after config freeze.
validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)
validation_tiles = validation_generator.tile_index.copy().reset_index(drop=True)
required = {"contig", "target_start", "target_end", "allowed_position_strand"}
missing = required - set(validation_tiles.columns)
assert not missing, ("validation", missing)
assert ((validation_tiles["target_end"] - validation_tiles["target_start"]) == profile_config.target_length).all()
assert (validation_tiles["target_start"].astype(np.int64) % REGION_SIZE == 0).all()
assert (validation_tiles["target_end"].astype(np.int64) % REGION_SIZE == 0).all()
assert int(validation_tiles["allowed_position_strand"].sum()) == EXPECTED_VALIDATION_POSITION_STRAND
assert int(validation_generator.total_positive) == EXPECTED_VALIDATION_POSITIVES

VALIDATION_RAW_DIR.mkdir(parents=False, exist_ok=False)

fixed_lowtp_val = pd.read_csv(FIXED_VALIDATION_LOW_TP)
fixed_hardfp_val = pd.read_csv(FIXED_VALIDATION_HARD_FP)
assert len(fixed_lowtp_val) == 14_052
assert len(fixed_hardfp_val) == 20_000

segment_names = ["overall", *VALIDATION_CONTIGS]
val_total_hist = {
    (model_name, segment): np.zeros(SUBMISSION_LEVELS, dtype=np.int64)
    for model_name in ("EXP045", "EXP046B") for segment in segment_names
}
val_pos_hist = {
    (model_name, segment): np.zeros(SUBMISSION_LEVELS, dtype=np.int64)
    for model_name in ("EXP045", "EXP046B") for segment in segment_names
}
cohort_bins = {
    (cohort, model_name): []
    for cohort in ("LOW_TP", "HARD_FP") for model_name in ("EXP045", "EXP046B")
}
validation_region_diag_rows = []
validation_cache_paths = []

val_started = time.perf_counter()
for contig in VALIDATION_CONTIGS:
    contig_tiles = validation_tiles[validation_tiles["contig"].astype(str) == contig].reset_index(drop=True)
    contig_code = CONTIG_TO_CODE[contig]
    lowtp_contig = fixed_lowtp_val[fixed_lowtp_val["contig"].astype(str) == contig]
    hardfp_contig = fixed_hardfp_val[fixed_hardfp_val["contig"].astype(str) == contig]

    region_keys = []
    region_risks = []
    region_active = []
    region_g = []
    raw_base_regions = []
    raw_valid_masks = []
    raw_positive_masks = []
    positions_seen = 0
    positives_seen = 0

    with torch.no_grad():
        for start_i in range(0, len(contig_tiles), EVAL_INFERENCE_BATCH_SIZE):
            selected = contig_tiles.iloc[start_i:start_i + EVAL_INFERENCE_BATCH_SIZE]
            batch = validation_generator.make_batch(selected)
            B = len(selected)
            assert B == 1
            x_gpu = batch.x_both_strands.to(device)
            feature_gpu = model.backbone(x_gpu)
            output_gpu = model.profile_head(feature_gpu)

            row = selected.iloc[0]
            target_start = int(row["target_start"])
            first_region_id = target_start // REGION_SIZE
            target_region_ids = first_region_id + np.arange(REGIONS_PER_TILE, dtype=np.int64)

            plus_count = batch.count[0, 0, :].numpy().astype(np.float32, copy=False)
            minus_count = batch.count[0, 1, :].numpy().astype(np.float32, copy=False)
            plus_mask = batch.mask[0, 0, :].numpy().astype(bool, copy=False)
            minus_mask = batch.mask[0, 1, :].numpy().astype(bool, copy=False)

            for strand_code, strand_name, orient_index, reverse_axis, count, mask in (
                (0, "+", 0, False, plus_count, plus_mask),
                (1, "-", 1, True, minus_count, minus_mask),
            ):
                stats, pool = regionalize_extended_orientation_cpu(
                    feature_gpu[orient_index], output_gpu[orient_index, 0], reverse_region_axis=reverse_axis
                )
                base_logit = genomic_target_base_logit(
                    output_gpu[orient_index, 0], reverse_base_axis=reverse_axis
                ).detach().cpu().numpy().astype(np.float32, copy=True)

                g = head_region_logits_from_extended(
                    stats, pool, final_head, SELECTED_VARIANT,
                    final_mean, final_std, final_prior_offset,
                )
                g_base = np.repeat(g, REGION_SIZE)
                final_logit = base_logit + np.float32(SELECTED_ALPHA) * g_base
                baseline_bins = probability_bin_5dp_from_logit(base_logit)
                final_bins = probability_bin_5dp_from_logit(final_logit)

                valid = mask
                positive = (count > 0) & mask
                positions_seen += int(valid.sum())
                positives_seen += int(positive.sum())
                for segment in ("overall", contig):
                    update_histogram(val_total_hist[("EXP045", segment)], baseline_bins[valid])
                    update_histogram(val_pos_hist[("EXP045", segment)], baseline_bins[positive])
                    update_histogram(val_total_hist[("EXP046B", segment)], final_bins[valid])
                    update_histogram(val_pos_hist[("EXP046B", segment)], final_bins[positive])

                valid_2d = valid.reshape(REGIONS_PER_TILE, REGION_SIZE)
                positive_2d = positive.reshape(REGIONS_PER_TILE, REGION_SIZE)
                valid_regions = valid_2d.any(axis=1)
                base_2d = base_logit.reshape(REGIONS_PER_TILE, REGION_SIZE)
                baseline_bins_2d = baseline_bins.reshape(REGIONS_PER_TILE, REGION_SIZE)
                risk = np.where(valid_2d, baseline_bins_2d, np.uint32(0)).max(axis=1).astype(np.uint32)
                active = positive_2d.any(axis=1)
                keys = pack_region_keys(contig_code, strand_code, target_region_ids)

                region_keys.append(keys[valid_regions])
                region_risks.append(risk[valid_regions])
                region_active.append(active[valid_regions])
                region_g.append(g[valid_regions])
                raw_base_regions.append(base_2d[valid_regions].copy())
                raw_valid_masks.append(bool_rows_to_u32(valid_2d[valid_regions]))
                raw_positive_masks.append(bool_rows_to_u32(positive_2d[valid_regions]))

                for cohort_name, frame in (("LOW_TP", lowtp_contig), ("HARD_FP", hardfp_contig)):
                    sub = frame[
                        (frame["strand"] == strand_name)
                        & (frame["position"] >= target_start)
                        & (frame["position"] < target_start + L)
                    ]
                    if len(sub):
                        local = (sub["position"].to_numpy(np.int64) - target_start).astype(np.int64)
                        cohort_bins[(cohort_name, "EXP045")].extend(baseline_bins[local].astype(np.int64).tolist())
                        cohort_bins[(cohort_name, "EXP046B")].extend(final_bins[local].astype(np.int64).tolist())

            del x_gpu, feature_gpu, output_gpu, batch, selected

    assert positions_seen == int(contig_tiles["allowed_position_strand"].sum())
    assert positives_seen > 0

    key_arr = np.concatenate(region_keys).astype(np.uint64, copy=False)
    risk_arr = np.concatenate(region_risks).astype(np.uint32, copy=False)
    active_arr = np.concatenate(region_active).astype(bool, copy=False)
    g_arr = np.concatenate(region_g).astype(np.float32, copy=False)
    base_arr = np.concatenate(raw_base_regions).astype(np.float32, copy=False)
    valid_mask_arr = np.concatenate(raw_valid_masks).astype(np.uint32, copy=False)
    pos_mask_arr = np.concatenate(raw_positive_masks).astype(np.uint32, copy=False)
    assert len(np.unique(key_arr)) == len(key_arr)

    cache_path = VALIDATION_RAW_DIR / f"{contig}_selected_region_cache.npz"
    np.savez(
        cache_path,
        region_key=key_arr,
        region_risk5=risk_arr,
        region_active=active_arr.astype(np.uint8),
        g=g_arr,
        base_logit=base_arr,
        valid_mask=valid_mask_arr,
        positive_mask=pos_mask_arr,
    )
    validation_cache_paths.append(cache_path)
    validation_region_diag_rows += regional_diagnostics(
        contig, SELECTED_VARIANT, g_arr, active_arr, risk_arr, key_arr, "final_validation"
    )
    print(
        f"[{contig}] positions={positions_seen:,} positives={positives_seen:,} "
        f"elapsed={(time.perf_counter()-val_started)/60:.1f} min"
    )

# Hard reproduction invariants before reading learned result.
assert int(val_total_hist[("EXP045", "overall")].sum()) == EXPECTED_VALIDATION_POSITION_STRAND
assert int(val_pos_hist[("EXP045", "overall")].sum()) == EXPECTED_VALIDATION_POSITIVES
baseline_ap5 = ap_from_histograms(val_pos_hist[("EXP045", "overall")], val_total_hist[("EXP045", "overall")])
assert abs(baseline_ap5 - EXPECTED_BASELINE_AP5) <= AP_TOL, baseline_ap5
for contig, expected in EXPECTED_VALIDATION_CONTIG_AP5.items():
    actual = ap_from_histograms(val_pos_hist[("EXP045", contig)], val_total_hist[("EXP045", contig)])
    assert abs(actual - expected) <= AP_TOL, (contig, actual, expected)

assert len(cohort_bins[("LOW_TP", "EXP045")]) == len(fixed_lowtp_val) == 14_052
assert len(cohort_bins[("LOW_TP", "EXP046B")]) == 14_052
assert len(cohort_bins[("HARD_FP", "EXP045")]) == len(fixed_hardfp_val) == 20_000
assert len(cohort_bins[("HARD_FP", "EXP046B")]) == 20_000

validation_region_diag_df = pd.DataFrame(validation_region_diag_rows)
validation_region_diag_df.to_csv(OUTPUTS["validation_region_diag"], index=False)
print("EXP045 final-validation reproduction: PASS")


In [ ]:
# ============================================================
# FINAL AP5 TABLE + FIXED-BUDGET RECALL + FIXED COHORT SUMMARY
# ============================================================
validation_metric_rows = []
for segment in segment_names:
    for model_name in ("EXP045", "EXP046B"):
        ap5 = ap_from_histograms(val_pos_hist[(model_name, segment)], val_total_hist[(model_name, segment)])
        validation_metric_rows.append({
            "segment": segment,
            "model": model_name,
            "ap5": ap5,
            "position_n": int(val_total_hist[(model_name, segment)].sum()),
            "positive_n": int(val_pos_hist[(model_name, segment)].sum()),
        })
validation_metrics_df = pd.DataFrame(validation_metric_rows)
validation_metrics_df.to_csv(OUTPUTS["validation_metrics"], index=False)

# Same top-K budgets for all positives and the fixed LOW_TP cohort.
budget_rows = []
for multiplier in BUDGET_MULTIPLIERS:
    budget = multiplier * EXPECTED_VALIDATION_POSITIVES
    for model_name in ("EXP045", "EXP046B"):
        total_hist = val_total_hist[(model_name, "overall")]
        positive_bins = np.repeat(np.arange(SUBMISSION_LEVELS, dtype=np.int64), val_pos_hist[(model_name, "overall")])
        positive_recall = tie_aware_recall_at_budget(total_hist, positive_bins, budget)
        lowtp_recall = tie_aware_recall_at_budget(
            total_hist, np.asarray(cohort_bins[("LOW_TP", model_name)], dtype=np.int64), budget
        )
        budget_rows.append({
            "budget_multiplier": multiplier,
            "budget_positions": int(budget),
            "model": model_name,
            "positive_position_recall": positive_recall,
            "fixed_lowtp_recall": lowtp_recall,
        })
budget_df = pd.DataFrame(budget_rows)
budget_df.to_csv(OUTPUTS["validation_budget"], index=False)

cohort_rows = []
for cohort_name in ("LOW_TP", "HARD_FP"):
    for model_name in ("EXP045", "EXP046B"):
        bins = np.asarray(cohort_bins[(cohort_name, model_name)], dtype=np.int64)
        cohort_rows.append({
            "cohort": cohort_name,
            "model": model_name,
            "n": int(len(bins)),
            "mean_score5": float(bins.mean() / 100_000.0),
            "median_score5": float(np.median(bins) / 100_000.0),
            "p90_score5": float(np.quantile(bins, 0.90) / 100_000.0),
            "p99_score5": float(np.quantile(bins, 0.99) / 100_000.0),
        })
cohort_df = pd.DataFrame(cohort_rows)
cohort_df.to_csv(OUTPUTS["validation_cohorts"], index=False)

print("Validation AP5:")
display(validation_metrics_df)
print("Same-budget recall:")
display(budget_df)
print("Fixed cohorts:")
display(cohort_df)


### Important implementation note on the fixed-budget table

The previous cell uses the existing 5-decimal submission protocol. At the cutoff score, ties are handled fractionally: if only a fraction of a tied bin fits inside top-K, the same fraction of cohort members in that bin is counted. This prevents arbitrary row order inside rounded-score ties from manufacturing a recall gain.


In [ ]:
# ============================================================
# RAW (UNROUNDED) AP AUDIT ON FINAL VALIDATION
# Per-contig always; overall attempted once for baseline and selected model.
# ============================================================
raw_validation_rows = []
for cache_path in validation_cache_paths:
    contig = cache_path.stem.replace("_selected_region_cache", "")
    data = np.load(cache_path, allow_pickle=False)
    base = data["base_logit"].astype(np.float32, copy=False)
    valid = u32_to_bool_rows(data["valid_mask"])
    positive = u32_to_bool_rows(data["positive_mask"])
    g = data["g"].astype(np.float32, copy=False)
    labels = positive[valid]
    baseline_scores = base[valid]
    final_scores = (base + np.float32(SELECTED_ALPHA) * g[:, None])[valid]
    raw_validation_rows.append({
        "segment": contig,
        "baseline_raw_ap": exact_ap_from_scores(baseline_scores, labels),
        "selected_raw_ap": exact_ap_from_scores(final_scores, labels),
    })
    del data, base, valid, positive, g, labels, baseline_scores, final_scores
    gc.collect()

# Overall exact AP: ~83.8M scores. Keep this audit outside selection and catch MemoryError.
def overall_raw_ap_from_validation_caches(selected):
    total_valid = 0
    for p in validation_cache_paths:
        d = np.load(p, allow_pickle=False)
        total_valid += int(u32_to_bool_rows(d["valid_mask"]).sum())
        del d
    assert total_valid == EXPECTED_VALIDATION_POSITION_STRAND
    scores = np.empty(total_valid, dtype=np.float32)
    labels = np.empty(total_valid, dtype=bool)
    cursor = 0
    for p in validation_cache_paths:
        d = np.load(p, allow_pickle=False)
        base = d["base_logit"].astype(np.float32, copy=False)
        valid = u32_to_bool_rows(d["valid_mask"])
        positive = u32_to_bool_rows(d["positive_mask"])
        if selected:
            g = d["g"].astype(np.float32, copy=False)
            local_scores = (base + np.float32(SELECTED_ALPHA) * g[:, None])[valid]
        else:
            local_scores = base[valid]
        local_labels = positive[valid]
        n = len(local_scores)
        scores[cursor:cursor+n] = local_scores
        labels[cursor:cursor+n] = local_labels
        cursor += n
        del d, base, valid, positive, local_scores, local_labels
    assert cursor == total_valid
    ap = exact_ap_from_scores(scores, labels)
    del scores, labels
    gc.collect()
    return ap

try:
    overall_baseline_raw = overall_raw_ap_from_validation_caches(selected=False)
    overall_selected_raw = overall_raw_ap_from_validation_caches(selected=True)
    raw_validation_rows.append({
        "segment": "overall",
        "baseline_raw_ap": overall_baseline_raw,
        "selected_raw_ap": overall_selected_raw,
    })
except MemoryError:
    warnings.warn(
        "Overall unrounded AP skipped after MemoryError. Per-contig raw AP remains available; "
        "the official selection/result still uses natural AP5."
    )
    raw_validation_rows.append({
        "segment": "overall",
        "baseline_raw_ap": float("nan"),
        "selected_raw_ap": float("nan"),
    })

raw_validation_df = pd.DataFrame(raw_validation_rows)
raw_validation_df["delta_raw_ap"] = raw_validation_df["selected_raw_ap"] - raw_validation_df["baseline_raw_ap"]
raw_validation_df.to_csv(RUN_DIR / "validation_raw_ap_audit.csv", index=False)
display(raw_validation_df)


In [ ]:
# ============================================================
# FINAL INTERPRETATION / DECISION — NO REGISTRY UPDATE HERE
# ============================================================
overall_table = validation_metrics_df[validation_metrics_df.segment == "overall"].set_index("model")
FINAL_BASELINE_AP5 = float(overall_table.loc["EXP045", "ap5"])
FINAL_AP5 = float(overall_table.loc["EXP046B", "ap5"])
DELTA_AP5 = FINAL_AP5 - FINAL_BASELINE_AP5

contig_wide = validation_metrics_df[validation_metrics_df.segment != "overall"].pivot(
    index="segment", columns="model", values="ap5"
)
contig_wide["delta"] = contig_wide["EXP046B"] - contig_wide["EXP045"]
ALL_CONTIGS_IMPROVED = bool((contig_wide["delta"] > 0).all())

budget_wide = budget_df.pivot(index="budget_multiplier", columns="model", values="fixed_lowtp_recall")
budget_wide["drop"] = budget_wide["EXP045"] - budget_wide["EXP046B"]
MAX_LOWTP_RECALL_DROP = float(np.maximum(budget_wide["drop"], 0.0).max())
LOWTP_GUARD_OK = bool(MAX_LOWTP_RECALL_DROP <= 0.05 + 1e-12)

if FINAL_AP5 < 0.11:
    decision = "REJECT"
elif FINAL_AP5 < 0.12:
    decision = "PROMISING" if ALL_CONTIGS_IMPROVED else "INCONCLUSIVE"
elif ALL_CONTIGS_IMPROVED and LOWTP_GUARD_OK:
    decision = "STRONG_CONFIRMATION" if FINAL_AP5 >= 0.15 else "ADOPT_FOR_END_TO_END_CHECK"
else:
    decision = "PROMISING_BUT_GUARDRAIL_FAILED"

# Plot final per-contig AP5.
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = validation_metrics_df[validation_metrics_df.segment != "overall"].pivot(
    index="segment", columns="model", values="ap5"
)
plot_df.plot(kind="bar", ax=ax)
ax.set_ylabel("natural nucleotide AP5")
ax.set_title(f"EXP046-B final validation — {SELECTED_VARIANT}, alpha={SELECTED_ALPHA}")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(OUTPUTS["validation_plot"], dpi=160)
plt.close(fig)

result_payload = {
    "experiment_id": EXPERIMENT_ID,
    "selected_from_train_cv": selected_payload,
    "final_validation": {
        "baseline_ap5": FINAL_BASELINE_AP5,
        "selected_ap5": FINAL_AP5,
        "delta_ap5": DELTA_AP5,
        "all_validation_contigs_improved": ALL_CONTIGS_IMPROVED,
        "max_fixed_lowtp_recall_drop_across_budgets": MAX_LOWTP_RECALL_DROP,
        "lowtp_guard_ok": LOWTP_GUARD_OK,
        "decision": decision,
    },
    "methodology": {
        "backbone_frozen": True,
        "nucleotide_head_frozen": True,
        "region_size": REGION_SIZE,
        "validation_used_for_training_or_mining": False,
        "selection_metric": "pooled train-contig OOF natural nucleotide AP5",
        "regional_auc_selection_metric": False,
        "raw_ap_used_for_selection": False,
        "raw_ap_role": "post-selection tie/ranking audit only",
    },
    "files": {name: str(path) for name, path in OUTPUTS.items()},
}
OUTPUTS["result_json"].write_text(
    json.dumps(result_payload, indent=2, ensure_ascii=False, allow_nan=True), encoding="utf-8"
)

print("=" * 88)
print("EXP046-B COMPLETE")
print("=" * 88)
print(f"Selected from train-CV: variant={SELECTED_VARIANT}, alpha={SELECTED_ALPHA}")
print(f"Train-CV pooled AP5: {SELECTED_CV_AP5:.8f}")
print(f"Validation AP5: {FINAL_BASELINE_AP5:.8f} -> {FINAL_AP5:.8f}  delta={DELTA_AP5:+.8f}")
print("All validation contigs improved:", ALL_CONTIGS_IMPROVED)
print("Max LOW_TP recall drop across fixed budgets:", MAX_LOWTP_RECALL_DROP)
print("Decision:", decision)
print("Result JSON:", OUTPUTS["result_json"])
print("\nRegistry/card intentionally NOT updated by this notebook.")


In [ ]:
# ============================================================
# CLEANUP TEMPORARY LARGE CACHES ONLY AFTER ALL RESULTS EXIST
# Fold models + final model + CSV/JSON/plots are retained.
# ============================================================
for required in (
    OUTPUTS["selected_config"], OUTPUTS["final_head"], OUTPUTS["validation_metrics"],
    OUTPUTS["validation_budget"], OUTPUTS["validation_region_diag"], OUTPUTS["result_json"],
):
    assert required.is_file(), f"Missing required result before cleanup: {required}"

for mm in (score_context_mm, pool_context_mm, filled_mm):
    mm.flush()
del score_context_mm, pool_context_mm, filled_mm
gc.collect()

# Keep the expensive fixed TRAIN sample feature cache: it was intentionally extracted once
# so a justified follow-up diagnostic can reuse it without another WIDTH512 pass.
# Raw CV/validation caches are much cheaper to reproduce and are deleted after raw-AP audits.
KEEP_FIXED_TRAIN_SAMPLE_CACHE = True
if CV_RAW_DIR.exists():
    shutil.rmtree(CV_RAW_DIR)
if VALIDATION_RAW_DIR.exists():
    shutil.rmtree(VALIDATION_RAW_DIR)
if KEEP_FIXED_TRAIN_SAMPLE_CACHE:
    print("Retained fixed train sample feature cache:", CACHE_DIR)
else:
    shutil.rmtree(CACHE_DIR)
    print("Deleted fixed train sample cache by explicit setting.")

print("EXP046-B finished without modifying EXP045/EXP046 oracle/frontier artifacts.")
